# Production Final Diffusion

Timestamp-driven final reconstruction: load production VGG/CLIP best weights, predict heldout tokens, run VGG-DIP low-level inversion, run CLIP-conditioned BrainIT/MindEye2 diffusion with best-of-10 candidate selection, export images, and evaluate all subjects.


## 1. Required Run Knobs


In [ ]:
RUN_SUBJECTS = ["subj02"]
FINAL_EVAL_IMAGES_PER_SUBJECT = 1000
MODEL_WEIGHTS_TIMESTAMP = "20261019_120000"  # Required: timestamp shared by production-vgg-train and production-clip-train.
SEED = 42


In [ ]:
# Modal Volume setup. Data is mounted and read like local files.
import os, re, json, random, time, gc, sys, subprocess, importlib.util, importlib.metadata, math
from collections import defaultdict, OrderedDict
from IPython.display import display

DATA_DIR = "/mnt/braid2-flatmap-data"
print("data dir:", DATA_DIR)
print("data dir exists:", os.path.isdir(DATA_DIR))
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(f"Mount the Modal Volume at {DATA_DIR}.")


def module_available(module_name):
    try:
        return importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        return False

_missing = []
for module_name, package_name in [
    ("h5py", "h5py"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("scipy", "scipy"),
    ("skimage", "scikit-image"),
    ("torch", "torch"),
    ("torchvision", "torchvision"),
    ("tqdm", "tqdm"),
    ("matplotlib", "matplotlib"),
]:
    if not module_available(module_name):
        _missing.append(package_name)
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.io import loadmat
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from torchvision import models

for _required_knob in ["RUN_SUBJECTS", "FINAL_EVAL_IMAGES_PER_SUBJECT", "MODEL_WEIGHTS_TIMESTAMP", "SEED"]:
    if _required_knob not in globals():
        raise RuntimeError(f"Run the key knobs cell first; missing {_required_knob}")

INPUT_DIR = DATA_DIR
FLATMAP_TOKENS_DIR = f"{DATA_DIR}/flatmap_tokens_fp16"
CLIP_TARGETS_DIR = f"{DATA_DIR}/clip_targets_cache"
LOCAL_OUTPUT_ROOT = f"{DATA_DIR}/outputs"
SHARED1000_EVAL_SIZE = 1000
FINAL_EVAL_IMAGES_PER_SUBJECT = min(int(FINAL_EVAL_IMAGES_PER_SUBJECT), SHARED1000_EVAL_SIZE)
if FINAL_EVAL_IMAGES_PER_SUBJECT < 1:
    raise ValueError("FINAL_EVAL_IMAGES_PER_SUBJECT must be at least 1")

def subject_run_tag(subjects):
    return "subj" + "".join(subject[-2:] for subject in subjects)

TRAIN_SUBJECTS = list(RUN_SUBJECTS)
PREDICTION_SUBJECTS = list(RUN_SUBJECTS)
FINAL_EVAL_SUBJECTS = list(RUN_SUBJECTS)
SUBJECT_TO_IDX = {subject: i for i, subject in enumerate(TRAIN_SUBJECTS)}
IDX_TO_SUBJECT = {i: subject for subject, i in SUBJECT_TO_IDX.items()}
NUM_SUBJECTS = len(TRAIN_SUBJECTS)
TRIALS_PER_SESSION = 750
KNOWN_BAD_SESSIONS = {("subj01", 11)}

if not str(MODEL_WEIGHTS_TIMESTAMP).strip():
    raise ValueError("Set MODEL_WEIGHTS_TIMESTAMP to the timestamp shared by the production VGG and CLIP training runs.")
SOURCE_RUN_TIMESTAMP = str(MODEL_WEIGHTS_TIMESTAMP).strip()
RUN_TIMESTAMP = SOURCE_RUN_TIMESTAMP
SPLIT_POLICY = "shared1000_image_id_holdout"
VAL_FRAC = 0.05
ARCH_MODE = "baseline"
OUTPUT_RUN_NAME = f"production_final_diffusion_subject02_baseline_{SPLIT_POLICY}_{ARCH_MODE}_{subject_run_tag(RUN_SUBJECTS)}_{RUN_TIMESTAMP}"
OUTPUT_DIR = f"{LOCAL_OUTPUT_ROOT}/{OUTPUT_RUN_NAME}"
VGG_OUTPUT_RUN_NAME = f"production_vgg_subject02_baseline_{SPLIT_POLICY}_{ARCH_MODE}_{subject_run_tag(RUN_SUBJECTS)}_{SOURCE_RUN_TIMESTAMP}"
CLIP_OUTPUT_RUN_NAME = f"production_clip_subject02_baseline_{SPLIT_POLICY}_{ARCH_MODE}_{subject_run_tag(RUN_SUBJECTS)}_{SOURCE_RUN_TIMESTAMP}"
VGG_RUN_DIR = f"{LOCAL_OUTPUT_ROOT}/{VGG_OUTPUT_RUN_NAME}"
CLIP_RUN_DIR = f"{LOCAL_OUTPUT_ROOT}/{CLIP_OUTPUT_RUN_NAME}"
VGG_BEST_WEIGHTS_PATH = f"{VGG_RUN_DIR}/vgg_best.pt"
CLIP_BEST_WEIGHTS_PATH = f"{CLIP_RUN_DIR}/clip_best.pt"
os.makedirs(OUTPUT_DIR, exist_ok=True)
for _label, _path in [("VGG best weights", VGG_BEST_WEIGHTS_PATH), ("CLIP best weights", CLIP_BEST_WEIGHTS_PATH)]:
    if not os.path.isfile(_path):
        raise FileNotFoundError(f"Missing {_label}: {_path}")
print("source VGG run:", VGG_RUN_DIR)
print("source CLIP run:", CLIP_RUN_DIR)

# BrainIT low-level branch settings from paper Appendix D.1/D.2.
VGG_IMAGE_SIZE = 112
VGG_TOKEN_DIM = 512
VGG_LAYER_NAMES = ["relu1_2", "relu2_2", "relu3_3", "relu4_3", "relu5_3"]
VGG_LAYER_TOKEN_COUNTS = [56 * 56, 55 * 55, 28 * 28, 14 * 14, 7 * 7]
VGG_LAYER_OFFSETS = np.cumsum([0] + VGG_LAYER_TOKEN_COUNTS[:-1]).tolist()
VGG_TOTAL_TOKENS = int(sum(VGG_LAYER_TOKEN_COUNTS))
SAMPLE_TOKENS_PER_LAYER = [512, 512, 128, 64, 16]
TEMPERATURE = 0.07

# OpenCLIP ViT-bigG/14 internal image patch tokens, matching bridge_train.py.
CLIP_DIM = 1664
CLIP_NUM_QUERIES = 256
CLIP_LOSS_WEIGHT = 1.0
VGG_LOSS_WEIGHT = 0.25

# Batch sizes and chunk sizes.
BATCH_SIZE = 128                     # VGG+CLIP training batch size
DIP_QUERY_BATCH = 1                    # shared-1000 VGG/CLIP prediction batch size
DIP_BATCH_SIZE = 36                    # DIP inversion batch size
DIP_MIN_BATCH_SIZE = 1                 # minimum DIP batch size after OOM backoff
FINAL_DIFFUSION_BATCH_SIZE = 50        # diffusion candidate batch size, divided by ensemble size
FINAL_EVAL_FEATURE_BATCH_SIZE = 32     # final eval feature-extraction batch size
PREDICT_QUERY_CHUNK = 1024             # VGG token query chunk size during prediction


D_MODEL = 768
N_LAYERS = 4
N_HEADS = 8
FFN_DIM = 3072
DROPOUT = 0.10
EPOCHS = 10
CACHE_TRAIN_SESSIONS_IN_RAM = False
LR = 5e-4
WEIGHT_DECAY = 1e-2
WARMUP_EPOCHS = 15
PLATEAU_PATIENCE = 2
PLATEAU_FACTOR = 0.1
GRAD_CLIP_NORM = 1.0
BRAIN_DIM_OVERRIDE = 768

DIP_INPUT_DIM = 32
DIP_INTERNAL_DIM = 128
DIP_SCALES = 3
DIP_STEPS = 2000
DIP_LR = 1e-2
DIP_INITIAL_NOISE = 0.1
DIP_REG_NOISE = 1.0 / 30.0
DIP_EMA = 0.99
DIP_OUTPUT_SIZE = 112
DIP_UPSAMPLED_SIZE = 256

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass
torch.backends.cudnn.benchmark = True


def ensure_xformers_wheel():
    if any(name == "xformers" or name.startswith("xformers.") for name in sys.modules):
        raise RuntimeError("Restart the kernel/container before repairing xformers; the current process already imported xformers.")

    torch_minor = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    cuda_version = torch.version.cuda
    cuda_tag = f"cu{cuda_version.replace('.', '')}" if cuda_version else None
    target_by_torch_cuda = {
        ("2.8", "12.9"): "0.0.32.post2",
    }
    target = target_by_torch_cuda.get((torch_minor, cuda_version))
    if target is None:
        raise RuntimeError(f"No xformers pin for torch {torch.__version__} / CUDA {cuda_version}; do not install xformers blindly.")

    try:
        installed = importlib.metadata.version("xformers")
    except importlib.metadata.PackageNotFoundError:
        installed = None

    if installed != target:
        print(f"installing xformers=={target} for torch {torch.__version__} / CUDA {cuda_version} (was {installed})")
        subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "xformers"])
        cmd = [
            sys.executable, "-m", "pip", "install",
            "--no-cache-dir", "--force-reinstall", "--no-deps",
            f"xformers=={target}",
        ]
        if cuda_tag:
            cmd += ["--index-url", f"https://download.pytorch.org/whl/{cuda_tag}"]
        subprocess.check_call(cmd)
    else:
        print(f"xformers=={target} already matches torch {torch.__version__} / CUDA {cuda_version}")

    subprocess.check_call([
        sys.executable, "-c",
        "import torch, xformers, xformers.ops; "
        "print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'xformers': xformers.__version__})",
    ])


ensure_xformers_wheel()


def torch_load_cpu(path, *args, **kwargs):
    kwargs.setdefault("map_location", "cpu")
    try:
        kwargs.setdefault("weights_only", False)
        return torch.load(path, *args, **kwargs)
    except TypeError:
        kwargs.pop("weights_only", None)
        return torch.load(path, *args, **kwargs)


def require_file(path):
    if not os.path.isfile(path):
        raise FileNotFoundError(path)
    return path

print(f"device={DEVICE} batch_size={BATCH_SIZE} output={OUTPUT_DIR}")
print(f"subjects={TRAIN_SUBJECTS}")
print(f"final_eval_images_per_subject={FINAL_EVAL_IMAGES_PER_SUBJECT} / {SHARED1000_EVAL_SIZE}")
print(f"model_weights_timestamp={MODEL_WEIGHTS_TIMESTAMP}")
print(f"VGG tokens={VGG_TOTAL_TOKENS} layer_counts={VGG_LAYER_TOKEN_COUNTS}")


## 2. Setup Code/All Data Loading


In [ ]:
EXP = require_file(f"{INPUT_DIR}/nsd_expdesign.mat")
mat = loadmat(EXP)

masterordering = mat["masterordering"].reshape(-1).astype(np.int64) - 1
subjectim = mat["subjectim"].astype(np.int64) - 1
shared_ids = set(mat["sharedix"].reshape(-1).astype(np.int64) - 1)

imgbrick_ids_by_subject = {}
for subject in TRAIN_SUBJECTS:
    subject_idx = int(subject[-2:]) - 1
    imgbrick_ids_by_subject[subject] = subjectim[subject_idx, masterordering]


def flatmap_candidates(subject, sess):
    return [
        f"{FLATMAP_TOKENS_DIR}/{subject}_session{sess:02d}_fp16.pt",
        f"{FLATMAP_TOKENS_DIR}/{subject}_flatmap_tokens_session{sess:02d}.pt",
    ]


def flatmap_read_path(subject, sess):
    for path in flatmap_candidates(subject, sess):
        if os.path.isfile(path):
            return path
    raise FileNotFoundError(f"Missing flatmap session {subject} {sess:02d}. Checked: {flatmap_candidates(subject, sess)}")


def discover_sessions(subject):
    if not os.path.isdir(FLATMAP_TOKENS_DIR):
        raise FileNotFoundError(FLATMAP_TOKENS_DIR)
    nums = []
    for name in os.listdir(FLATMAP_TOKENS_DIR):
        if not name.startswith(subject):
            continue
        match = re.search(r"session(\d+)", name)
        if match:
            nums.append(int(match.group(1)))
    nums = sorted(set(s for s in nums if (subject, s) not in KNOWN_BAD_SESSIONS))
    if len(nums) < 2:
        raise RuntimeError(f"{subject} needs at least 2 usable sessions, found {nums}")
    return nums


session_nums_by_subject = {subject: discover_sessions(subject) for subject in TRAIN_SUBJECTS}
train_pairs = []
val_pairs = []
for subject in TRAIN_SUBJECTS:
    sessions = session_nums_by_subject[subject]
    train_pairs.extend((subject, sess) for sess in sessions)
    val_pairs.extend((subject, sess) for sess in sessions)
    print(f"{subject}: {len(sessions)} sessions train={sessions} val={sessions}")
print(f"total: {len(train_pairs)} train sessions, {len(val_pairs)} val sessions")


def load_flatmap_tokens(subject, sess):
    brain_tokens = torch_load_cpu(flatmap_read_path(subject, sess))
    if brain_tokens.dtype != torch.float16:
        brain_tokens = brain_tokens.half()
    if not torch.isfinite(brain_tokens).all():
        bad = int((~torch.isfinite(brain_tokens)).sum().item())
        raise ValueError(f"{subject} session {sess:02d} has {bad} non-finite flatmap token values")
    return brain_tokens.contiguous()


def ids_for_session(subject, sess, n_rows):
    start = (sess - 1) * TRIALS_PER_SESSION
    return imgbrick_ids_by_subject[subject][start:start + n_rows].astype(np.int64)


FINAL_HELDOUT_IMG_IDS = set(int(i) for i in shared_ids)
VAL_HELDOUT_IMG_IDS_BY_SUBJECT = {}
for subject in TRAIN_SUBJECTS:
    nonshared_img_ids = np.array(sorted({
        int(img_id)
        for sess in session_nums_by_subject[subject]
        for img_id in ids_for_session(subject, sess, TRIALS_PER_SESSION)
        if int(img_id) not in FINAL_HELDOUT_IMG_IDS
    }), dtype=np.int64)
    image_perm = np.random.RandomState(SEED + SUBJECT_TO_IDX[subject]).permutation(nonshared_img_ids)
    n_val_images = int(VAL_FRAC * len(image_perm))
    VAL_HELDOUT_IMG_IDS_BY_SUBJECT[subject] = set(map(int, image_perm[:n_val_images]))
VAL_HELDOUT_IMG_IDS_ALL = set().union(*VAL_HELDOUT_IMG_IDS_BY_SUBJECT.values())
TRAIN_BLOCKED_IMG_IDS_BY_SUBJECT = {
    subject: FINAL_HELDOUT_IMG_IDS | VAL_HELDOUT_IMG_IDS_ALL
    for subject in TRAIN_SUBJECTS
}


def split_keep_mask(subject, img_ids, split):
    ids = [int(i) for i in img_ids.tolist()]
    if split == "train":
        blocked = TRAIN_BLOCKED_IMG_IDS_BY_SUBJECT[subject]
        keep = [img_id not in blocked for img_id in ids]
    elif split == "val":
        heldout = VAL_HELDOUT_IMG_IDS_BY_SUBJECT[subject]
        keep = [img_id in heldout for img_id in ids]
    elif split in {"all", "eval"}:
        keep = [True] * len(ids)
    else:
        raise ValueError(f"unknown split={split!r}")
    return torch.as_tensor(keep, dtype=torch.bool)


def apply_split_filter(subject, brain_tokens, img_ids, split):
    keep = split_keep_mask(subject, img_ids, split)
    return brain_tokens[keep], img_ids[keep]


def assert_clean_image_id_splits():
    for subject in TRAIN_SUBJECTS:
        train_seen = set()
        for train_subject, sess in train_pairs:
            if train_subject != subject:
                continue
            ids = torch.from_numpy(ids_for_session(subject, sess, TRIALS_PER_SESSION)).long()
            keep = split_keep_mask(subject, ids, "train")
            train_seen.update(int(i) for i in ids[keep].tolist())
        val_seen = set()
        for val_subject, sess in val_pairs:
            if val_subject != subject:
                continue
            ids = torch.from_numpy(ids_for_session(subject, sess, TRIALS_PER_SESSION)).long()
            keep = split_keep_mask(subject, ids, "val")
            val_seen.update(int(i) for i in ids[keep].tolist())
        assert train_seen.isdisjoint(FINAL_HELDOUT_IMG_IDS), f"{subject}: train/final image-id overlap"
        assert train_seen.isdisjoint(VAL_HELDOUT_IMG_IDS_ALL), f"{subject}: train/global-val image-id overlap"
        assert val_seen.isdisjoint(FINAL_HELDOUT_IMG_IDS), f"{subject}: val/final image-id overlap"
        assert train_seen.isdisjoint(val_seen), f"{subject}: train/val image-id overlap"
        print(
            f"{subject}: clean image-id split train_unique={len(train_seen)} "
            f"val_unique={len(val_seen)} final_heldout={len(FINAL_HELDOUT_IMG_IDS)}"
        )


assert_clean_image_id_splits()


def infer_flatmap_shape():
    subject, sess = train_pairs[0]
    sample = load_flatmap_tokens(subject, sess)
    if sample.ndim != 3:
        raise ValueError(f"expected flatmap tokens [trials, patches, dim], got {tuple(sample.shape)}")
    return int(sample.shape[1]), int(sample.shape[2])


NUM_FLATMAP_PATCHES, FLATMAP_TOKEN_DIM = infer_flatmap_shape()
print(f"flatmap direct tokens: patches={NUM_FLATMAP_PATCHES} dim={FLATMAP_TOKEN_DIM}")


## 3. Image Loader + BrainIT VGG Tokenizer


In [ ]:
STIM_PATH = require_file(f"{INPUT_DIR}/nsd_stimuli.hdf5")
_H5 = None
_IMG_DS = None


def stimulus_dataset():
    global _H5, _IMG_DS
    if _IMG_DS is None:
        _H5 = h5py.File(STIM_PATH, "r")
        for key in ["imgBrick", "images", "stimuli"]:
            if key in _H5:
                _IMG_DS = _H5[key]
                break
        if _IMG_DS is None:
            raise KeyError(f"Could not find stimulus image dataset. HDF5 keys={list(_H5.keys())}")
    return _IMG_DS


def preprocess_images_uint8(images_np, size):
    if images_np.ndim != 4:
        raise ValueError(f"expected NHWC or NCHW images, got {images_np.shape}")
    if images_np.shape[-1] == 3:
        img = torch.from_numpy(images_np).permute(0, 3, 1, 2)
    elif images_np.shape[1] == 3:
        img = torch.from_numpy(images_np)
    else:
        raise ValueError(f"could not infer image channels from {images_np.shape}")
    img = img.float()
    if img.max() > 2:
        img = img / 255.0
    return F.interpolate(img, size=(size, size), mode="bicubic", align_corners=False).clamp(0, 1)


def images_for_ids(img_ids, size=VGG_IMAGE_SIZE):
    ids = np.asarray([int(i) for i in img_ids], dtype=np.int64)
    unique_ids, inverse = np.unique(ids, return_inverse=True)
    imgs_unique = stimulus_dataset()[unique_ids]
    imgs = imgs_unique[inverse]
    return preprocess_images_uint8(imgs, size)


class BrainITVGGTokenizer(nn.Module):
    """VGG16-BN feature maps and tokenization described in BrainIT Appendix D.2."""
    def __init__(self):
        super().__init__()
        weights = models.VGG16_BN_Weights.DEFAULT
        self.features = models.vgg16_bn(weights=weights).features.eval()
        self.capture = {5: 0, 12: 1, 22: 2, 32: 3, 42: 4}
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        for p in self.parameters():
            p.requires_grad_(False)

    @staticmethod
    def _repeat_to_512(tokens):
        if tokens.shape[-1] == VGG_TOKEN_DIM:
            return tokens
        reps = math.ceil(VGG_TOKEN_DIM / tokens.shape[-1])
        return tokens.repeat(1, 1, reps)[..., :VGG_TOKEN_DIM]

    def extract_features(self, images01):
        x = F.interpolate(images01, size=(VGG_IMAGE_SIZE, VGG_IMAGE_SIZE), mode="bicubic", align_corners=False).clamp(0, 1)
        x = (x - self.mean) / self.std
        outs = [None] * len(VGG_LAYER_NAMES)
        for i, layer in enumerate(self.features):
            x = layer(x)
            if i in self.capture:
                outs[self.capture[i]] = x
        return outs

    def _tokens_from_feature(self, feat, layer_idx):
        if layer_idx == 0:
            tokens = F.unfold(feat, kernel_size=2, stride=2).transpose(1, 2)
        elif layer_idx == 1:
            tokens = F.unfold(feat, kernel_size=2, stride=1).transpose(1, 2)
        else:
            tokens = feat.flatten(2).transpose(1, 2)
        return self._repeat_to_512(tokens).contiguous()

    def tokens_from_features(self, features):
        return [self._tokens_from_feature(feat, i) for i, feat in enumerate(features)]

    def forward(self, images01):
        return self.tokens_from_features(self.extract_features(images01))

vgg_tokenizer = BrainITVGGTokenizer().to(DEVICE).eval()
print("VGG tokenizer ready:", VGG_LAYER_NAMES)


## 4. Define Full Model Architecture


In [ ]:

class CrossAttentionBridge(nn.Module):
    def __init__(self, brain_dim, target_dim, num_queries, d_model, n_layers, n_heads, ffn_dim, dropout):
        super().__init__()
        self.query_embed = nn.Parameter(torch.randn(num_queries, d_model) * 0.02)
        self.brain_proj = nn.Linear(brain_dim, d_model) if brain_dim != d_model else nn.Identity()
        self.brain_norm = nn.LayerNorm(d_model)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ffn_dim,
            dropout=dropout,
            batch_first=True,
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(d_model, target_dim)

    def forward(self, brain_tokens, query_indices=None):
        b = brain_tokens.shape[0]
        memory = self.brain_norm(self.brain_proj(brain_tokens))
        if query_indices is None:
            queries = self.query_embed.unsqueeze(0).expand(b, -1, -1)
        elif query_indices.ndim == 1:
            queries = self.query_embed[query_indices].unsqueeze(0).expand(b, -1, -1)
        else:
            queries = self.query_embed[query_indices]
        out = self.decoder(tgt=queries, memory=memory)
        return self.output_proj(out)


class FlatmapSingleHeadBridge(nn.Module):
    def __init__(self, brain_dim, target):
        super().__init__()
        if target not in {"vgg", "clip"}:
            raise ValueError(f"target must be 'vgg' or 'clip', got {target!r}")
        self.target = target
        self.num_patches = int(NUM_FLATMAP_PATCHES)
        if target == "vgg":
            target_dim = VGG_TOKEN_DIM
            num_queries = VGG_TOTAL_TOKENS
        else:
            target_dim = CLIP_DIM
            num_queries = CLIP_NUM_QUERIES
        self.bridge = CrossAttentionBridge(
            brain_dim=brain_dim,
            target_dim=target_dim,
            num_queries=num_queries,
            d_model=D_MODEL,
            n_layers=N_LAYERS,
            n_heads=N_HEADS,
            ffn_dim=FFN_DIM,
            dropout=DROPOUT,
        )

    def forward(self, brain_tokens, query_indices=None):
        return self.bridge(brain_tokens, query_indices=query_indices)


model = None
brain_dim = int(FLATMAP_TOKEN_DIM)


def initialize_fresh_model(target, label=None):
    global model, brain_dim
    if target not in {"vgg", "clip"}:
        raise ValueError(f"target must be 'vgg' or 'clip', got {target!r}")
    brain_dim = int(FLATMAP_TOKEN_DIM)
    model = FlatmapSingleHeadBridge(brain_dim, target=target).to(DEVICE)
    params_m = sum(p.numel() for p in model.parameters()) / 1e6
    label = label or target
    if target == "vgg":
        target_desc = f"VGG [{VGG_TOTAL_TOKENS}, {VGG_TOKEN_DIM}]"
    else:
        target_desc = f"CLIP [{CLIP_NUM_QUERIES}, {CLIP_DIM}]"
    print(f"initialized {label}: brain_dim={brain_dim} target={target_desc} params={params_m:.1f}M")
    print(f"single-subject direct flatmap baseline architecture: {NUM_FLATMAP_PATCHES} patches -> transformer memory tokens")
    return model


## 5. Shared Prediction Helpers


In [ ]:
# Shared prediction/evaluation helpers copied from the production training stack.

def sample_layer_indices(train=True):
    query_ids, splits = [], []
    for offset, n_tokens, k in zip(VGG_LAYER_OFFSETS, VGG_LAYER_TOKEN_COUNTS, SAMPLE_TOKENS_PER_LAYER):
        k = min(k, n_tokens)
        if train:
            ids = torch.randperm(n_tokens, device=DEVICE)[:k]
        else:
            ids = torch.linspace(0, n_tokens - 1, steps=k, device=DEVICE).long()
        query_ids.append(ids + offset)
        splits.append(k)
    return torch.cat(query_ids), splits


def select_target_tokens(tokens_by_layer, query_ids, splits):
    chunks = []
    start = 0
    for li, (offset, k) in enumerate(zip(VGG_LAYER_OFFSETS, splits)):
        local_ids = query_ids[start:start + k] - offset
        chunks.append(tokens_by_layer[li][:, local_ids])
        start += k
    return torch.cat(chunks, dim=1)


def info_nce_by_position(pred, target):
    b, k, _ = pred.shape
    if b < 2:
        return F.mse_loss(pred, target)
    pred = F.normalize(pred.float(), dim=-1)
    target = F.normalize(target.float(), dim=-1)
    logits = torch.einsum("bkd,ckd->kbc", pred, target) / TEMPERATURE
    labels = torch.arange(b, device=pred.device).repeat(k)
    loss_i = F.cross_entropy(logits.reshape(k * b, b), labels)
    loss_t = F.cross_entropy(logits.transpose(1, 2).reshape(k * b, b), labels)
    return 0.5 * (loss_i + loss_t)


def split_layers(x, splits):
    out, start = [], 0
    for k in splits:
        out.append(x[:, start:start + k])
        start += k
    return out


def vgg_contrastive_loss(pred, target, splits):
    loss = pred.new_tensor(0.0)
    parts = {}
    for name, p, t in zip(VGG_LAYER_NAMES, split_layers(pred, splits), split_layers(target, splits)):
        layer_loss = info_nce_by_position(p, t)
        parts[f"nce_{name}"] = layer_loss
        loss = loss + layer_loss
    return loss / len(splits), parts


def feature_metrics(pred, target):
    return {
        "mse": float(F.mse_loss(pred.float(), target.float()).detach().cpu()),
        "cos": float(F.cosine_similarity(pred.float().flatten(1), target.float().flatten(1), dim=1).mean().detach().cpu()),
    }


clip_targets_by_subject = {}


def get_clip_targets(subject):
    if subject not in clip_targets_by_subject:
        path = require_file(f"{CLIP_TARGETS_DIR}/{subject}_clip_targets.pt")
        clip_targets_by_subject[subject] = torch_load_cpu(path)
        print(f"loaded CLIP targets {subject}: {len(clip_targets_by_subject[subject])} images")
    return clip_targets_by_subject[subject]


def clip_targets_for_ids(subject, img_ids):
    clip_by_imgbrick = get_clip_targets(subject)
    targets = torch.stack([clip_by_imgbrick[int(img_id)] for img_id in img_ids]).contiguous()
    expected = (CLIP_NUM_QUERIES, CLIP_DIM)
    if tuple(targets.shape[1:]) != expected:
        raise ValueError(f"expected CLIP targets [B, {expected[0]}, {expected[1]}], got {tuple(targets.shape)}")
    if not torch.isfinite(targets).all():
        bad = int((~torch.isfinite(targets)).sum().item())
        raise ValueError(f"CLIP target batch has {bad} non-finite values")
    return targets


session_cache = {}
bad_pairs = set()


def load_session(subject, sess, split="train"):
    key = (subject, int(sess), split)
    if CACHE_TRAIN_SESSIONS_IN_RAM and key in session_cache:
        return session_cache[key]
    bad_key = (subject, int(sess))
    if bad_key in bad_pairs:
        return None
    try:
        brain_tokens = load_flatmap_tokens(subject, sess)
        ids = torch.from_numpy(ids_for_session(subject, sess, brain_tokens.shape[0])).long()
        brain_tokens, ids = apply_split_filter(subject, brain_tokens, ids, split)
        if brain_tokens.shape[0] == 0:
            return None
        loaded = (brain_tokens, ids)
        if CACHE_TRAIN_SESSIONS_IN_RAM:
            session_cache[key] = loaded
        return loaded
    except Exception as exc:
        print(f"[bad session] {subject} session {sess:02d}: {exc}; skipping")
        bad_pairs.add(bad_key)
        return None


def assert_checkpoint_split_policy(ckpt, path):
    if not isinstance(ckpt, dict):
        raise ValueError(f"refusing to load legacy bare checkpoint without split metadata: {path}")
    if ckpt.get("split_policy") != SPLIT_POLICY:
        raise ValueError(
            f"refusing to load checkpoint with split_policy={ckpt.get('split_policy')!r}; "
            f"expected {SPLIT_POLICY!r}: {path}"
        )
    if float(ckpt.get("val_frac", VAL_FRAC)) != float(VAL_FRAC):
        raise ValueError(
            f"refusing to load checkpoint with val_frac={ckpt.get('val_frac')!r}; "
            f"expected {VAL_FRAC!r}: {path}"
        )

    expected_arch_token = "direct_flatmap"
    arch = str(ckpt.get("architecture", ""))
    if expected_arch_token not in arch:
        raise ValueError(
            f"refusing to load checkpoint with architecture={arch!r}; "
            f"expected architecture containing {expected_arch_token!r}: {path}"
        )


## 6. Predict Shared-1000 VGG + CLIP Tokens From Best Weights


In [ ]:
@torch.no_grad()
def predict_all_vgg_tokens(brain_batch, subject):
    model.eval()
    brain_batch = brain_batch.to(DEVICE, non_blocking=True).float()
    outs = []
    for start in range(0, VGG_TOTAL_TOKENS, PREDICT_QUERY_CHUNK):
        ids = torch.arange(start, min(start + PREDICT_QUERY_CHUNK, VGG_TOTAL_TOKENS), device=DEVICE)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            outs.append(model(brain_batch, query_indices=ids).float().cpu())
    return torch.cat(outs, dim=1)


@torch.no_grad()
def predict_clip_tokens(brain_batch, subject):
    model.eval()
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        return model(brain_batch.to(DEVICE, non_blocking=True).float()).float().cpu()


def split_all_vgg_tokens(x):
    out, start = [], 0
    for n in VGG_LAYER_TOKEN_COUNTS:
        out.append(x[:, start:start + n])
        start += n
    return out


def shared_tokens_path(subject):
    return f"{OUTPUT_DIR}/{subject}_shared1000_flatmap_source_tokens.pt"


def shared_ids_path(subject):
    return f"{OUTPUT_DIR}/{subject}_shared1000_img_ids.json"


def pred_vgg_path(subject):
    return f"{OUTPUT_DIR}/{subject}_shared1000_transfer_pred_vgg_tokens.pt"


def pred_clip_path(subject):
    return f"{OUTPUT_DIR}/{subject}_shared1000_transfer_pred_clip_tokens.pt"


def collect_shared1000_flatmap_tokens(subject):
    tokens_path = shared_tokens_path(subject)
    ids_path = shared_ids_path(subject)
    if os.path.exists(tokens_path) and os.path.exists(ids_path):
        with open(ids_path) as f:
            ids = json.load(f)
        if any(int(img_id) not in FINAL_HELDOUT_IMG_IDS for img_id in ids):
            raise ValueError(f"{subject} cached shared1000 IDs include non-heldout image IDs")
        return torch_load_cpu(tokens_path), ids
    groups = defaultdict(list)
    shared_id_set = set(int(i) for i in shared_ids)
    for sess in tqdm(session_nums_by_subject[subject], desc=f"collect shared-1000 flatmaps {subject}"):
        brain_tokens = load_flatmap_tokens(subject, sess)
        sess_ids = ids_for_session(subject, sess, brain_tokens.shape[0])
        for row, img_id in enumerate(sess_ids):
            if int(img_id) in shared_id_set:
                groups[int(img_id)].append(brain_tokens[row])
        del brain_tokens
        gc.collect()
    ids = sorted(groups)
    if any(int(img_id) not in FINAL_HELDOUT_IMG_IDS for img_id in ids):
        raise ValueError(f"{subject} shared1000 collection included non-heldout image IDs")
    avg_tokens = torch.stack([torch.stack(groups[img_id]).mean(0) for img_id in ids]).half().contiguous()
    torch.save(avg_tokens, tokens_path)
    with open(ids_path, "w") as f:
        json.dump([int(i) for i in ids], f)
    return avg_tokens, ids


def checkpoint_for_head(head):
    if head == "vgg":
        return VGG_BEST_WEIGHTS_PATH
    if head == "clip":
        return CLIP_BEST_WEIGHTS_PATH
    raise ValueError(f"unknown head={head!r}")


def load_prediction_checkpoint(path, head):
    initialize_fresh_model(head, f"{head} best prediction")
    ckpt = torch_load_cpu(path, map_location=DEVICE)
    assert_checkpoint_split_policy(ckpt, path)
    if isinstance(ckpt, dict) and ckpt.get("domain") != head:
        raise ValueError(f"checkpoint domain={ckpt.get('domain')!r} does not match requested head={head!r}: {path}")
    model.load_state_dict(ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt)
    print("loaded best prediction checkpoint:", path)


def predict_shared1000_for_subject(subject):
    shared_brain_tokens, shared_img_ids = collect_shared1000_flatmap_tokens(subject)
    print(f"{subject} shared flatmap source tokens:", tuple(shared_brain_tokens.shape), "ids:", len(shared_img_ids))
    vgg_path = pred_vgg_path(subject)
    clip_path = pred_clip_path(subject)

    if not os.path.exists(vgg_path):
        load_prediction_checkpoint(checkpoint_for_head("vgg"), "vgg")
        vgg_preds = []
        for start in tqdm(range(0, len(shared_brain_tokens), DIP_QUERY_BATCH), desc=f"predict shared-1000 VGG tokens {subject}"):
            brain_batch = shared_brain_tokens[start:start + DIP_QUERY_BATCH]
            vgg_preds.append(predict_all_vgg_tokens(brain_batch, subject))
        pred_vgg_tokens = torch.cat(vgg_preds, dim=0).half().contiguous()
        torch.save(pred_vgg_tokens, vgg_path)
        print(f"{subject}: saved predicted VGG tokens:", tuple(pred_vgg_tokens.shape), vgg_path)
        del pred_vgg_tokens, vgg_preds
        gc.collect()
    else:
        print(f"{subject}: predicted VGG exists:", vgg_path)

    if not os.path.exists(clip_path):
        load_prediction_checkpoint(checkpoint_for_head("clip"), "clip")
        clip_preds = []
        for start in tqdm(range(0, len(shared_brain_tokens), DIP_QUERY_BATCH), desc=f"predict shared-1000 CLIP tokens {subject}"):
            brain_batch = shared_brain_tokens[start:start + DIP_QUERY_BATCH]
            clip_preds.append(predict_clip_tokens(brain_batch, subject))
        pred_clip_tokens = torch.cat(clip_preds, dim=0).half().contiguous()
        torch.save(pred_clip_tokens, clip_path)
        print(f"{subject}: saved predicted CLIP tokens:", tuple(pred_clip_tokens.shape), clip_path)
        del pred_clip_tokens, clip_preds
        gc.collect()
    else:
        print(f"{subject}: predicted CLIP exists:", clip_path)

    del shared_brain_tokens
    gc.collect()
    return vgg_path, clip_path


prediction_manifest = {}
for subject in PREDICTION_SUBJECTS:
    vgg_path, clip_path = predict_shared1000_for_subject(subject)
    prediction_manifest[subject] = {
        "pred_vgg_path": vgg_path,
        "pred_clip_path": clip_path,
        "shared_tokens_path": shared_tokens_path(subject),
        "shared_ids_path": shared_ids_path(subject),
        "source_vgg_best_weights_path": VGG_BEST_WEIGHTS_PATH,
        "source_clip_best_weights_path": CLIP_BEST_WEIGHTS_PATH,
    }
with open(f"{OUTPUT_DIR}/final_diffusion_prediction_manifest.json", "w") as f:
    json.dump(prediction_manifest, f, indent=2)


def load_subject_prediction_for_lowlevel(subject):
    with open(shared_ids_path(subject)) as f:
        ids = json.load(f)
    return {
        "subject": subject,
        "shared_img_ids": ids,
        "pred_vgg_tokens": torch_load_cpu(pred_vgg_path(subject)),
        "pred_clip_tokens": torch_load_cpu(pred_clip_path(subject)),
    }


for subject in PREDICTION_SUBJECTS:
    subject_prediction = load_subject_prediction_for_lowlevel(subject)
    print(
        "prediction subject:", subject,
        "predicted VGG:", tuple(subject_prediction["pred_vgg_tokens"].shape),
        "predicted CLIP:", tuple(subject_prediction["pred_clip_tokens"].shape),
        "ids:", len(subject_prediction["shared_img_ids"]),
    )


## 7. VGG-DIP Low-Level Inversion


In [ ]:
class GroupedConvBlock(nn.Module):
    def __init__(self, groups, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(groups * in_ch, groups * out_ch, 3, padding=1, groups=groups),
            nn.BatchNorm2d(groups * out_ch),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(groups * out_ch, groups * out_ch, 3, padding=1, groups=groups),
            nn.BatchNorm2d(groups * out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class BatchedDIPUNet(nn.Module):
    def __init__(self, groups, input_dim=DIP_INPUT_DIM, width=DIP_INTERNAL_DIM):
        super().__init__()
        self.groups = int(groups)
        self.input_dim = int(input_dim)
        self.width = int(width)
        self.down1 = GroupedConvBlock(groups, input_dim, width)
        self.down2 = GroupedConvBlock(groups, width, width)
        self.down3 = GroupedConvBlock(groups, width, width)
        self.mid = GroupedConvBlock(groups, width, width)
        self.up3 = GroupedConvBlock(groups, width * 2, width)
        self.up2 = GroupedConvBlock(groups, width * 2, width)
        self.up1 = GroupedConvBlock(groups, width * 2, width)
        self.out = nn.Conv2d(groups * width, groups * 3, 1, groups=groups)

    def _merge(self, x):
        b, c, h, w = x.shape
        if b != self.groups:
            raise ValueError(f"expected batch={self.groups}, got {b}")
        return x.reshape(1, b * c, h, w)

    def _split(self, x, channels):
        _, _, h, w = x.shape
        return x.reshape(self.groups, channels, h, w)

    def forward(self, z):
        z = self._merge(z)
        e1 = self.down1(z)
        e2 = self.down2(F.avg_pool2d(e1, 2))
        e3 = self.down3(F.avg_pool2d(e2, 2))
        x = self.mid(F.avg_pool2d(e3, 2))
        x = F.interpolate(x, size=e3.shape[-2:], mode="bilinear", align_corners=False)
        x = self.up3(torch.cat([x, e3], dim=1))
        x = F.interpolate(x, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        x = self.up2(torch.cat([x, e2], dim=1))
        x = F.interpolate(x, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        return self._split(torch.sigmoid(self.out(x)), 3)


def reduce_repeated_token_dim(tokens, true_dim):
    if tokens.shape[-1] == true_dim:
        return tokens
    if tokens.shape[-1] % true_dim != 0:
        return tokens[..., :true_dim]
    return tokens.reshape(*tokens.shape[:-1], tokens.shape[-1] // true_dim, true_dim).mean(dim=-2)


def fold_overlapping_tokens(tokens, channels, output_hw, kernel_size, stride):
    b = tokens.shape[0]
    cols = tokens.transpose(1, 2).contiguous()
    feat = F.fold(cols, output_size=output_hw, kernel_size=kernel_size, stride=stride)
    ones = torch.ones((1, channels * kernel_size * kernel_size, tokens.shape[1]), device=tokens.device, dtype=tokens.dtype)
    counts = F.fold(ones, output_size=output_hw, kernel_size=kernel_size, stride=stride).clamp_min(1e-6)
    return feat / counts


def tokens_to_brainit_vgg_features(tokens):
    layers = split_all_vgg_tokens(tokens.float())
    feat1_tokens = reduce_repeated_token_dim(layers[0], 4 * 64)
    feat2_tokens = reduce_repeated_token_dim(layers[1], 4 * 128)
    feat3_tokens = reduce_repeated_token_dim(layers[2], 256)
    feat4_tokens = reduce_repeated_token_dim(layers[3], 512)
    feat5_tokens = reduce_repeated_token_dim(layers[4], 512)

    feat1 = fold_overlapping_tokens(feat1_tokens, channels=64, output_hw=(112, 112), kernel_size=2, stride=2)
    feat2 = fold_overlapping_tokens(feat2_tokens, channels=128, output_hw=(56, 56), kernel_size=2, stride=1)
    feat3 = feat3_tokens.transpose(1, 2).reshape(tokens.shape[0], 256, 28, 28).contiguous()
    feat4 = feat4_tokens.transpose(1, 2).reshape(tokens.shape[0], 512, 14, 14).contiguous()
    feat5 = feat5_tokens.transpose(1, 2).reshape(tokens.shape[0], 512, 7, 7).contiguous()
    return [feat1, feat2, feat3, feat4, feat5]


DIP_LAYER_WEIGHTS = [1.0, 1.0, 1.0, 1.0, 1.0]
DIP_COMPILE = False
DIP_PARALLEL_GPUS = globals().get("DIP_PARALLEL_GPUS", True)
DIP_GPU_IDS = globals().get("DIP_GPU_IDS", None)  # None means all visible CUDA devices.
DIP_BATCH_SIZE_PER_GPU = int(globals().get("DIP_BATCH_SIZE_PER_GPU", DIP_BATCH_SIZE))

import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

DIP_DATA_LOCK = threading.Lock()


def canonical_torch_device(device):
    device = torch.device(device)
    if device.type == "cuda" and device.index is None:
        return torch.device(f"cuda:{torch.cuda.current_device()}")
    return device


def dip_loss(pred_img, target_features, tokenizer=None):
    tokenizer = vgg_tokenizer if tokenizer is None else tokenizer
    pred_features = tokenizer.extract_features(pred_img)
    loss = pred_img.new_tensor(0.0)
    for weight, pred, target in zip(DIP_LAYER_WEIGHTS, pred_features, target_features):
        loss = loss + weight * F.mse_loss(pred.float(), target.float())
    return loss / sum(DIP_LAYER_WEIGHTS)


def maybe_compile_dip(dip):
    if not DIP_COMPILE or not hasattr(torch, "compile"):
        return dip
    try:
        return torch.compile(dip, mode="reduce-overhead")
    except Exception as exc:
        print(f"torch.compile skipped: {exc}")
        return dip


def invert_vgg_features_with_dip(target_features, desc="DIP", device=DEVICE, tokenizer=None, generator=None):
    batch_size = int(target_features[0].shape[0])
    device = canonical_torch_device(device)
    tokenizer = vgg_tokenizer if tokenizer is None else tokenizer
    dip = BatchedDIPUNet(batch_size).to(device).train()
    dip = maybe_compile_dip(dip)
    z = torch.randn(
        batch_size,
        DIP_INPUT_DIM,
        DIP_OUTPUT_SIZE,
        DIP_OUTPUT_SIZE,
        device=device,
        generator=generator,
    ) * DIP_INITIAL_NOISE
    opt = torch.optim.Adam(dip.parameters(), lr=DIP_LR)
    ema = None
    target_features = [t.to(device).float().detach() for t in target_features]

    pbar = tqdm(range(DIP_STEPS), desc=desc, leave=False, mininterval=2.0)
    for step in pbar:
        z_noise = torch.randn(z.shape, device=device, dtype=z.dtype, generator=generator)
        z_in = z + z_noise * DIP_REG_NOISE
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=device.type == "cuda"):
            img = dip(z_in)
            loss = dip_loss(img, target_features, tokenizer=tokenizer)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        with torch.no_grad():
            current = dip(z).detach()
            ema = current if ema is None else DIP_EMA * ema + (1.0 - DIP_EMA) * current
        if step % 100 == 0:
            pbar.set_postfix(loss=f"{float(loss.detach().cpu()):.4f}")

    lowlevel_112 = ema.clamp(0, 1)
    lowlevel_256 = F.interpolate(lowlevel_112, size=(DIP_UPSAMPLED_SIZE, DIP_UPSAMPLED_SIZE), mode="bicubic", align_corners=False).clamp(0, 1)
    return lowlevel_112.cpu(), lowlevel_256.cpu()


DIP_VERSION = "v2"
DIP_SAMPLE_SEED = SEED
RERUN_DIP_INVERSION = False
DIP_USE_GROUND_TRUTH_VGG = False

DIP_TARGET_SOURCE = "gtvgg" if DIP_USE_GROUND_TRUTH_VGG else "predvgg"
DIP_OUTPUT_TAG = f"{DIP_VERSION}_gtvgg" if DIP_USE_GROUND_TRUTH_VGG else DIP_VERSION


def dip_paths_for_subject(subject):
    recon_dir = f"{OUTPUT_DIR}/{subject}_shared1000_dip_recon_shards_{DIP_OUTPUT_TAG}"
    os.makedirs(recon_dir, exist_ok=True)
    return {
        "recon_dir": recon_dir,
        "recons_path": f"{OUTPUT_DIR}/{subject}_shared1000_dip_recons_{DIP_OUTPUT_TAG}.pt",
        "recons112_path": f"{OUTPUT_DIR}/{subject}_shared1000_dip_recons_112_{DIP_OUTPUT_TAG}.pt",
        "img_ids_path": f"{OUTPUT_DIR}/{subject}_shared1000_dip_img_ids_{DIP_OUTPUT_TAG}.json",
        "source_indices_path": f"{OUTPUT_DIR}/{subject}_shared1000_dip_source_indices_{DIP_OUTPUT_TAG}.json",
    }


def dip_shard_path(paths, i):
    return f"{paths['recon_dir']}/recon_{i:04d}.pt"


def dip_target_features_for_indices(subject, batch_indices, shared_img_ids, pred_vgg_tokens, device=DEVICE, tokenizer=None):
    device = canonical_torch_device(device)
    tokenizer = vgg_tokenizer if tokenizer is None else tokenizer
    if DIP_USE_GROUND_TRUTH_VGG:
        img_ids = [int(shared_img_ids[i]) for i in batch_indices]
        with DIP_DATA_LOCK:
            images = images_for_ids(img_ids, size=VGG_IMAGE_SIZE).to(device, non_blocking=True)
        with torch.no_grad():
            target_features = tokenizer.extract_features(images)
        target_features = [feat.detach().cpu() for feat in target_features]
        del images
        return target_features
    return tokens_to_brainit_vgg_features(pred_vgg_tokens[batch_indices])


def final_eval_indices_for_subject(subject, n_available):
    n_requested = min(FINAL_EVAL_IMAGES_PER_SUBJECT, n_available)
    if n_requested >= n_available:
        return list(range(n_available))
    seed = DIP_SAMPLE_SEED + SUBJECT_TO_IDX[subject]
    rng = np.random.RandomState(seed)
    return sorted(rng.choice(n_available, size=n_requested, replace=False).astype(int).tolist())


def run_dip_inversion_for_subject(subject, device=DEVICE, tokenizer=None):
    device = canonical_torch_device(device)
    tokenizer = vgg_tokenizer if tokenizer is None else tokenizer
    if device.type == "cuda":
        torch.cuda.set_device(device)
    dip_generator = torch.Generator(device=device)
    dip_generator.manual_seed(DIP_SAMPLE_SEED + SUBJECT_TO_IDX[subject])
    active_prediction = load_subject_prediction_for_lowlevel(subject)
    shared_img_ids = active_prediction["shared_img_ids"]
    pred_vgg_tokens = active_prediction["pred_vgg_tokens"]
    paths = dip_paths_for_subject(subject)
    print("DIP subject:", subject, "device:", str(device), "predicted VGG:", tuple(pred_vgg_tokens.shape), "ids:", len(shared_img_ids))

    n_available = min(SHARED1000_EVAL_SIZE, len(shared_img_ids), pred_vgg_tokens.shape[0])
    recon_indices = final_eval_indices_for_subject(subject, n_available)
    pending_indices = [i for i in recon_indices if RERUN_DIP_INVERSION or not os.path.exists(dip_shard_path(paths, i))]
    active_batch_size = min(DIP_BATCH_SIZE_PER_GPU, max(len(pending_indices), 1))
    print(
        f"DIP images to invert for {subject}: {len(pending_indices)} pending / {len(recon_indices)} requested / "
        f"{n_available} available, batch_size_per_gpu={active_batch_size}, rerun={RERUN_DIP_INVERSION}, target={DIP_TARGET_SOURCE}"
    )
    start = 0
    pbar = tqdm(total=len(pending_indices), desc=f"DIP invert batches {subject}")
    while start < len(pending_indices):
        batch_indices = pending_indices[start:start + active_batch_size]
        target_features = recon112 = recon256 = None
        try:
            target_features = dip_target_features_for_indices(subject, batch_indices, shared_img_ids, pred_vgg_tokens, device=device, tokenizer=tokenizer)
            recon112, recon256 = invert_vgg_features_with_dip(
                target_features,
                desc=f"DIP {subject} {DIP_TARGET_SOURCE} {device} batch {start // active_batch_size + 1}",
                device=device,
                tokenizer=tokenizer,
                generator=dip_generator,
            )
            for j, i in enumerate(batch_indices):
                torch.save({
                    "img_id": int(shared_img_ids[i]),
                    "source_index": int(i),
                    "recon112": recon112[j].half(),
                    "recon": recon256[j].half(),
                    "target_source": DIP_TARGET_SOURCE,
                }, dip_shard_path(paths, i))
            start += len(batch_indices)
            pbar.update(len(batch_indices))
            del target_features, recon112, recon256
        except RuntimeError as exc:
            del target_features, recon112, recon256
            if device.type == "cuda" and "out of memory" in str(exc).lower() and active_batch_size > DIP_MIN_BATCH_SIZE:
                active_batch_size = max(DIP_MIN_BATCH_SIZE, active_batch_size // 2)
                print(f"CUDA OOM; retrying with DIP batch_size={active_batch_size}")
            else:
                raise
        gc.collect()
        if device.type == "cuda":
            with torch.cuda.device(device):
                torch.cuda.empty_cache()
    pbar.close()

    recons112, recons256, ids_done, source_indices_done = [], [], [], []
    for i in recon_indices:
        item = torch_load_cpu(dip_shard_path(paths, i))
        ids_done.append(int(item["img_id"]))
        source_indices_done.append(int(item.get("source_index", i)))
        recons112.append(item["recon112"].float())
        recons256.append(item["recon"].float())
    all_recons112 = torch.stack(recons112).clamp(0, 1)
    all_recons = torch.stack(recons256).clamp(0, 1)
    torch.save(all_recons112, paths["recons112_path"])
    torch.save(all_recons, paths["recons_path"])
    with open(paths["img_ids_path"], "w") as f:
        json.dump(ids_done, f)
    with open(paths["source_indices_path"], "w") as f:
        json.dump(source_indices_done, f)
    print("saved DIP recons 112:", tuple(all_recons112.shape), paths["recons112_path"])
    print("saved DIP recons 256:", tuple(all_recons.shape), paths["recons_path"])
    return {"subject": subject, "n_requested": len(recon_indices), "n_available": n_available, **paths}


def dip_worker_devices():
    if DEVICE.type != "cuda" or not DIP_PARALLEL_GPUS or torch.cuda.device_count() <= 1:
        return [DEVICE]
    if DIP_GPU_IDS is None:
        gpu_ids = list(range(torch.cuda.device_count()))
    else:
        gpu_ids = [int(i) for i in DIP_GPU_IDS]
    if not gpu_ids:
        return [DEVICE]
    return [torch.device(f"cuda:{i}") for i in gpu_ids]


def make_dip_tokenizer_for_device(device):
    device = canonical_torch_device(device)
    try:
        existing_device = next(vgg_tokenizer.parameters()).device
    except StopIteration:
        existing_device = vgg_tokenizer.mean.device
    if device == existing_device:
        return vgg_tokenizer
    if device.type == "cuda":
        with torch.cuda.device(device):
            return BrainITVGGTokenizer().to(device).eval()
    return BrainITVGGTokenizer().to(device).eval()


def run_dip_subject_group(subjects, device, tokenizer):
    results = {}
    for subject in subjects:
        results[subject] = run_dip_inversion_for_subject(subject, device=device, tokenizer=tokenizer)
    return results


dip_devices = dip_worker_devices()
if len(dip_devices) == 1 or len(FINAL_EVAL_SUBJECTS) <= 1:
    dip_device = dip_devices[0]
    dip_tokenizer = make_dip_tokenizer_for_device(dip_device)
    print("DIP inversion device:", str(dip_device), "batch_size_per_gpu:", DIP_BATCH_SIZE_PER_GPU)
    dip_manifest = {
        subject: run_dip_inversion_for_subject(subject, device=dip_device, tokenizer=dip_tokenizer)
        for subject in FINAL_EVAL_SUBJECTS
    }
else:
    subject_groups = [list(FINAL_EVAL_SUBJECTS[i::len(dip_devices)]) for i in range(len(dip_devices))]
    worker_specs = [(device, group) for device, group in zip(dip_devices, subject_groups) if group]
    print(
        "DIP inversion devices:",
        {str(device): group for device, group in worker_specs},
        "batch_size_per_gpu:", DIP_BATCH_SIZE_PER_GPU,
        "effective_parallel_batch:", DIP_BATCH_SIZE_PER_GPU * len(worker_specs),
    )
    dip_tokenizers = {str(device): make_dip_tokenizer_for_device(device) for device, _ in worker_specs}
    dip_manifest = {}
    with ThreadPoolExecutor(max_workers=len(worker_specs)) as executor:
        futures = [
            executor.submit(run_dip_subject_group, group, device, dip_tokenizers[str(device)])
            for device, group in worker_specs
        ]
        for future in as_completed(futures):
            dip_manifest.update(future.result())
    dip_manifest = {subject: dip_manifest[subject] for subject in FINAL_EVAL_SUBJECTS}

DIP_MANIFEST_PATH = f"{OUTPUT_DIR}/dip_prediction_manifest_{DIP_OUTPUT_TAG}.json"
with open(DIP_MANIFEST_PATH, "w") as f:
    json.dump(dip_manifest, f, indent=2)
print("saved DIP manifest:", DIP_MANIFEST_PATH)


## 8. VGG-DIP Preview


In [ ]:
from skimage.color import rgb2gray
from skimage.metrics import structural_similarity as ssim_fn

PREVIEW_GRID_N = 48
PREVIEW_GRID_COLS = 6


def eval_dip_subject(subject, paths):
    eval_path = f"{OUTPUT_DIR}/{subject}_shared1000_dip_eval_{DIP_OUTPUT_TAG}.csv"
    preview_path = f"{OUTPUT_DIR}/{subject}_shared1000_dip_preview_grid_{DIP_OUTPUT_TAG}.png"

    all_recons = torch_load_cpu(paths["recons_path"]).float().clamp(0, 1)
    with open(paths["img_ids_path"]) as f:
        ids_done = json.load(f)
    ids_done = ids_done[:len(all_recons)]
    true_images = images_for_ids(ids_done, size=DIP_UPSAMPLED_SIZE).float()

    pixel_mse = F.mse_loss(all_recons, true_images).item()
    pixcorr_scores, ssim_scores = [], []
    for rec, true in tqdm(list(zip(all_recons, true_images)), desc=f"eval DIP recons {subject}"):
        rec_np = rec.permute(1, 2, 0).numpy()
        true_np = true.permute(1, 2, 0).numpy()
        pixcorr_scores.append(np.corrcoef(rec_np.reshape(-1), true_np.reshape(-1))[0, 1])
        ssim_scores.append(ssim_fn(rgb2gray(true_np), rgb2gray(rec_np), data_range=1.0))

    row = {
        "subject": subject,
        "N": len(all_recons),
        "Pixel_MSE": pixel_mse,
        "PixCorr": float(np.nanmean(pixcorr_scores)),
        "SSIM": float(np.nanmean(ssim_scores)),
        "target_source": DIP_TARGET_SOURCE,
    }
    pd.DataFrame([row]).to_csv(eval_path, index=False)
    print(pd.DataFrame([row]))
    print("saved:", eval_path)

    n_show = min(PREVIEW_GRID_N, len(all_recons))
    show_idx = np.arange(n_show)
    cols = min(PREVIEW_GRID_COLS, n_show)
    rows = int(np.ceil(n_show / cols))
    combined = torch.cat([true_images[show_idx], all_recons[show_idx]], dim=3).clamp(0, 1)
    fig, axes = plt.subplots(rows, cols, figsize=(4.8 * cols, 3.0 * rows), constrained_layout=True)
    axes = np.atleast_1d(axes).reshape(rows, cols)
    for ax in axes.ravel():
        ax.axis("off")
    for k, idx in enumerate(show_idx):
        ax = axes[k // cols, k % cols]
        ax.imshow(combined[k].permute(1, 2, 0))
        ax.set_title(f"{subject} shared id {ids_done[int(idx)]}\nactual | VGG-DIP", fontsize=8)
    plt.savefig(preview_path, dpi=180, bbox_inches="tight")
    plt.show()
    print("saved:", preview_path)
    return row


if "dip_manifest" not in globals():
    with open(f"{OUTPUT_DIR}/dip_prediction_manifest_{DIP_OUTPUT_TAG}.json") as f:
        dip_manifest = json.load(f)

eval_rows = [eval_dip_subject(subject, dip_manifest[subject]) for subject in FINAL_EVAL_SUBJECTS]
eval_summary = pd.DataFrame(eval_rows)
EVAL_SUMMARY_PATH = f"{OUTPUT_DIR}/final_dip_eval_summary_{DIP_OUTPUT_TAG}.csv"
eval_summary.to_csv(EVAL_SUMMARY_PATH, index=False)
print(eval_summary)
print("saved summary:", EVAL_SUMMARY_PATH)


## 9. BrainIT/MindEye2 Diffusion Setup


In [ ]:
# BrainIT/MindEye2 diffusion setup copied from final-eval and adapted to the single-head notebook artifacts.
import warnings, logging, importlib, importlib.metadata

DIFFUSION_START_STEP = 14
DIFFUSION_NUM_STEPS = 38
DIFFUSION_ENSEMBLE_SIZE = 10
SAVE_DIFFUSION_ENSEMBLE_CANDIDATES = True
FINAL_DIFFUSION_AUTO_SPLIT_ON_OOM = True
FINAL_DIFFUSION_EMPTY_CACHE_EVERY = 20
USE_CLIP_TOKENS_FOR_DIFFUSION = True
USE_LOWLEVEL_IMAGE_INIT_FOR_DIFFUSION = True
RUN_BRAINIT_SDXL_ENHANCEMENT = False
RERUN_FINAL_DIFFUSION = False
DIFFUSION_EXPERIMENT_NAME = ""
RUN_SWAV_EVAL = True

BRAINIT_DIR = f"{DATA_DIR}/brainit-fmri"
BRAINIT_EXTERNAL_MODEL_DIR = f"{BRAINIT_DIR}/data/external_models"
MINDEYE2_MODEL_DIR = os.environ.get("MINDEYE2_MODEL_DIR", f"{BRAINIT_EXTERNAL_MODEL_DIR}/MindEyeV2")
MINDEYE2_UNCLIP_CKPT_PATH = os.environ.get("MINDEYE2_UNCLIP_CKPT_PATH", f"{MINDEYE2_MODEL_DIR}/unclip6_epoch0_step110000.ckpt")
MINDEYE2_UNCLIP_CONFIG = f"{BRAINIT_DIR}/src/MindEyeV2/generative_models/configs/unclip6.yaml"
BRAINIT_SDXL_CONFIG = f"{BRAINIT_DIR}/src/MindEyeV2/generative_models/configs/inference/sd_xl_base.yaml"
BRAINIT_SDXL_CKPT_PATH = os.environ.get("BRAINIT_SDXL_CKPT_PATH", f"{BRAINIT_EXTERNAL_MODEL_DIR}/SDXL/zavychromaxl_v30.safetensors")
BRAINIT_SAVED_MODELS_DIR = f"{BRAINIT_DIR}/results/saved_models"
BRAINIT_COMBINED_CKPT_PATH = os.environ.get("BRAINIT_COMBINED_CKPT_PATH", f"{BRAINIT_SAVED_MODELS_DIR}/combined_model.ckpt")
BRAINIT_CLIPG_MODEL_PATH = os.environ.get("BRAINIT_CLIPG_MODEL_PATH", f"{BRAINIT_SAVED_MODELS_DIR}/decoder_clipg_ext-1_save.pth")

for module_name, package_name in [
    ("packaging", "packaging"),
    ("diffusers", "diffusers"),
    ("accelerate", "accelerate"),
    ("transformers", "transformers"),
    ("safetensors", "safetensors"),
    ("omegaconf", "omegaconf"),
    ("pytorch_lightning", "pytorch-lightning"),
    ("einops", "einops"),
    ("open_clip", "open_clip_torch==2.24.0"),
    ("clip", "clip-anytorch"),
    ("timm", "timm"),
    ("torch_geometric", "torch-geometric"),
    ("kornia", "kornia"),
    ("webdataset", "webdataset"),
    ("huggingface_hub", "huggingface_hub"),
]:
    if not module_available(module_name):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])


def ensure_package_version(dist_name, package_spec, version, purge_modules=()):
    try:
        installed = importlib.metadata.version(dist_name)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        print(f"installing {package_spec} for BrainIT compatibility (was {installed})")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", package_spec])
        for name in list(sys.modules):
            if name in purge_modules or any(name.startswith(prefix + ".") for prefix in purge_modules):
                del sys.modules[name]


def disable_inference_checkpointing(engine):
    candidates = [engine, getattr(engine, "model", None)]
    model_ = getattr(engine, "model", None)
    candidates.append(getattr(model_, "diffusion_model", None))
    disabled = False
    for obj in candidates:
        fn = getattr(obj, "disable_gradient_checkpointing", None) if obj is not None else None
        if callable(fn):
            fn()
            disabled = True
    return disabled

require_file(MINDEYE2_UNCLIP_CONFIG)
require_file(MINDEYE2_UNCLIP_CKPT_PATH)
require_file(BRAINIT_COMBINED_CKPT_PATH)
require_file(BRAINIT_CLIPG_MODEL_PATH)
if RUN_BRAINIT_SDXL_ENHANCEMENT:
    require_file(BRAINIT_SDXL_CONFIG)
    require_file(BRAINIT_SDXL_CKPT_PATH)
if DEVICE.type != "cuda":
    raise RuntimeError("BrainIT diffusion uses CUDA-only calls. Run this cell on a GPU runtime.")

warnings.filterwarnings("ignore", message=r"`torch\.cuda\.amp\.autocast.*deprecated.*", category=FutureWarning)
warnings.filterwarnings("ignore", message=r"torch\.utils\.checkpoint: the use_reentrant parameter should be passed explicitly.*", category=UserWarning)
warnings.filterwarnings("ignore", message=r"None of the inputs have requires_grad=True.*", category=UserWarning)

# Match BrainIT's import layout: repo root must beat src/MindEyeV2/utils.py and src/MindEyeV2/models.py.
os.chdir(BRAINIT_DIR)
brainit_paths = [
    BRAINIT_DIR,
    f"{BRAINIT_DIR}/src",
    f"{BRAINIT_DIR}/src/MindEyeV2",
    f"{BRAINIT_DIR}/src/MindEyeV2/generative_models",
]
sys.path[:] = [p for p in sys.path if p not in brainit_paths]
sys.path[:0] = brainit_paths
for name in list(sys.modules):
    purge = (
        name == "utils" or name.startswith("utils.")
        or name == "models" or name.startswith("models.")
        or name == "open_clip" or name.startswith("open_clip.")
        or name == "generative_models" or name.startswith("generative_models.")
        or name == "sgm" or name.startswith("sgm.")
        or name.startswith("src.MindEyeV2.generative_models")
    )
    if purge:
        del sys.modules[name]
importlib.invalidate_caches()

import utils as brainit_utils_pkg
import models as brainit_models_pkg
if not hasattr(brainit_utils_pkg, "__path__") or not any(os.path.abspath(p).startswith(os.path.abspath(f"{BRAINIT_DIR}/utils")) for p in brainit_utils_pkg.__path__):
    raise RuntimeError(f"Wrong utils import: {getattr(brainit_utils_pkg, '__file__', brainit_utils_pkg)}")
if not hasattr(brainit_models_pkg, "__path__") or not any(os.path.abspath(p).startswith(os.path.abspath(f"{BRAINIT_DIR}/models")) for p in brainit_models_pkg.__path__):
    raise RuntimeError(f"Wrong models import: {getattr(brainit_models_pkg, '__file__', brainit_models_pkg)}")
print("BrainIT utils package:", list(brainit_utils_pkg.__path__))
print("BrainIT models package:", list(brainit_models_pkg.__path__))

try:
    import xformers
    import xformers.ops
except Exception as e:
    raise RuntimeError("xformers failed to import. Rerun the Modal setup cell so it can install the Torch/CUDA-matched xformers wheel.") from e

q = torch.randn(1, 8, 64, 64, device=DEVICE, dtype=torch.float16)
k = torch.randn_like(q)
v = torch.randn_like(q)
_ = xformers.ops.memory_efficient_attention(q, k, v)
del q, k, v, _

ensure_package_version("open-clip-torch", "open_clip_torch==2.24.0", "2.24.0", purge_modules=("open_clip", "generative_models", "sgm", "src.MindEyeV2.generative_models"))

if not module_available("clip"):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "clip-anytorch"])
try:
    import pkg_resources  # noqa: F401
    import packaging
    if not hasattr(pkg_resources, "packaging"):
        pkg_resources.packaging = packaging
except Exception:
    import types, packaging
    pkg_resources = types.ModuleType("pkg_resources")
    pkg_resources.packaging = packaging
sys.modules["pkg_resources"] = pkg_resources
import clip

for module_name, package_name in [("timm", "timm"), ("torch_geometric", "torch-geometric")]:
    if not module_available(module_name):
        print(f"installing {package_name} for BrainIT checkpoint loading...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
import timm  # noqa: F401
import torch_geometric  # noqa: F401

try:
    from diffusers.models.vae import Decoder  # noqa: F401
except Exception:
    import types
    from diffusers.models.autoencoders.vae import Decoder
    vae_compat = types.ModuleType("diffusers.models.vae")
    vae_compat.Decoder = Decoder
    sys.modules["diffusers.models.vae"] = vae_compat

import diffusers, transformers, pytorch_lightning
logging.getLogger("sgm.modules.attention").setLevel(logging.ERROR)
print("runtime:", {
    "torch": torch.__version__,
    "diffusers": diffusers.__version__,
    "transformers": transformers.__version__,
    "pytorch_lightning": pytorch_lightning.__version__,
    "xformers": getattr(xformers, "__version__", "installed"),
    "clip_file": getattr(clip, "__file__", "unknown"),
})

from utils.diffusion_utils import load_diffusion_engine, enhance_recons
from models.combined_diffusion_engine import CombinedDiffusionEngine
from generative_models.sgm.modules.encoders import modules as sgm_encoder_modules


def patch_openclip_text_encoder_batch_first():
    def is_batch_first(model_):
        transformer = getattr(model_, "transformer", None)
        if bool(getattr(transformer, "batch_first", False)):
            return True
        blocks = getattr(transformer, "resblocks", [])
        return bool(blocks and getattr(getattr(blocks[0], "attn", None), "batch_first", False))

    def forward_blocks(self, x, attn_mask=None):
        outputs = {}
        batch_first = is_batch_first(self.model)
        blocks = self.model.transformer.resblocks
        for i, block in enumerate(blocks):
            if i == len(blocks) - 1:
                outputs["penultimate"] = x if batch_first else x.permute(1, 0, 2)
            if self.model.transformer.grad_checkpointing and not torch.jit.is_scripting():
                x = torch.utils.checkpoint.checkpoint(block, x, attn_mask, use_reentrant=False)
            else:
                x = block(x, attn_mask=attn_mask)
        outputs["last"] = x if batch_first else x.permute(1, 0, 2)
        return outputs

    def encode2(self, text):
        x = self.model.token_embedding(text) + self.model.positional_embedding
        if not is_batch_first(self.model):
            x = x.permute(1, 0, 2)
        outputs = self.text_transformer_forward(x, attn_mask=self.model.attn_mask)
        if self.legacy:
            z = self.model.ln_final(outputs[self.layer])
            return z
        last = self.model.ln_final(outputs["last"])
        pooled = last[torch.arange(last.shape[0], device=last.device), text.argmax(dim=-1)] @ self.model.text_projection
        outputs["last"] = last
        outputs["pooled"] = pooled
        return outputs

    def encode1(self, text):
        x = self.model.token_embedding(text) + self.model.positional_embedding
        if not is_batch_first(self.model):
            x = x.permute(1, 0, 2)
        for i, block in enumerate(self.model.transformer.resblocks):
            if i == len(self.model.transformer.resblocks) - self.layer_idx:
                break
            if self.model.transformer.grad_checkpointing and not torch.jit.is_scripting():
                x = torch.utils.checkpoint.checkpoint(block, x, self.model.attn_mask, use_reentrant=False)
            else:
                x = block(x, attn_mask=self.model.attn_mask)
        if not is_batch_first(self.model):
            x = x.permute(1, 0, 2)
        return self.model.ln_final(x)

    sgm_encoder_modules.FrozenOpenCLIPEmbedder2.text_transformer_forward = forward_blocks
    sgm_encoder_modules.FrozenOpenCLIPEmbedder2.encode_with_transformer = encode2
    sgm_encoder_modules.FrozenOpenCLIPEmbedder.encode_with_transformer = encode1
    print("patched BrainIT OpenCLIP text encoders for batch_first compatibility")


patch_openclip_text_encoder_batch_first()

class PrecomputedClipAdapter(nn.Module):
    def forward(self, fmri, indexes=None, edge_index_v2c=None):
        return fmri

print("loading MindEye2 unCLIP diffusion engine...")
base_diffusion_engine = load_diffusion_engine(
    config_path=MINDEYE2_UNCLIP_CONFIG,
    ckpt_path=MINDEYE2_UNCLIP_CKPT_PATH,
)
print("gradient checkpointing disabled:", disable_inference_checkpointing(base_diffusion_engine))

print("loading BrainIT CLIP-G decoder for checkpoint structure...")
brainit_clipg_model = torch_load_cpu(BRAINIT_CLIPG_MODEL_PATH, map_location="cpu")
print("loading BrainIT combined diffusion checkpoint...")
combined_model = CombinedDiffusionEngine.load_from_checkpoint(
    BRAINIT_COMBINED_CKPT_PATH,
    map_location="cpu",
    weights_only=False,
    diffusion_engine=base_diffusion_engine,
    gnn_model=brainit_clipg_model,
)
combined_model.gnn_model = PrecomputedClipAdapter()
combined_model = combined_model.to(DEVICE).eval().requires_grad_(False)
combined_model.diffusion_engine.sampler.num_steps = DIFFUSION_NUM_STEPS
print("BrainIT enhancement:", RUN_BRAINIT_SDXL_ENHANCEMENT)
print("diffusion ready")


## 10. Diffuse Low-Level DIP Images Using CLIP Top-10 Ensemble


In [ ]:
# Diffuse each subject's predicted CLIP tokens, initialized from its low-level DIP images.
# Edit these here while tuning throughput. FINAL_DIFFUSION_BATCH_SIZE is total candidates per model call.
DIFFUSION_ENSEMBLE_SIZE = 10
FINAL_DIFFUSION_BATCH_SIZE = 40
DIFFUSION_ENSEMBLE_SIZE = max(1, int(DIFFUSION_ENSEMBLE_SIZE))
FINAL_DIFFUSION_BATCH_SIZE = max(1, int(FINAL_DIFFUSION_BATCH_SIZE))
FINAL_RECON_VARIANT = f"s{DIFFUSION_START_STEP}_n{DIFFUSION_NUM_STEPS}"
if not USE_LOWLEVEL_IMAGE_INIT_FOR_DIFFUSION:
    FINAL_RECON_VARIANT += "_no_lowlevel"
if not USE_CLIP_TOKENS_FOR_DIFFUSION:
    FINAL_RECON_VARIANT += "_no_clip"
if DIFFUSION_ENSEMBLE_SIZE > 1:
    FINAL_RECON_VARIANT += f"_bestof{DIFFUSION_ENSEMBLE_SIZE}_lowlevel_mse"
if not RUN_BRAINIT_SDXL_ENHANCEMENT:
    FINAL_RECON_VARIANT += "_no_sdxl"
if DIFFUSION_EXPERIMENT_NAME:
    FINAL_RECON_VARIANT += "_" + str(DIFFUSION_EXPERIMENT_NAME).strip().replace(" ", "_").replace("/", "-")
FINAL_RECON_TAG = f"singlehead_finaleval{FINAL_EVAL_IMAGES_PER_SUBJECT}_{FINAL_RECON_VARIANT}"


def _same_ids(a, b):
    return [int(x) for x in a] == [int(x) for x in b]


def _rows_for_ids(source_ids, wanted_ids, label):
    source_ids = [int(i) for i in source_ids]
    wanted_ids = [int(i) for i in wanted_ids]
    lookup = {img_id: row for row, img_id in enumerate(source_ids)}
    missing = [img_id for img_id in wanted_ids if img_id not in lookup]
    if missing:
        raise ValueError(f"{label} is missing {len(missing)} requested image ids; first missing ids={missing[:8]}")
    return torch.as_tensor([lookup[img_id] for img_id in wanted_ids], dtype=torch.long)


def final_diffusion_paths_for_subject(subject):
    semantic_recon_dir = f"{OUTPUT_DIR}/{subject}_brainit_semantic_uint8_{FINAL_RECON_TAG}"
    os.makedirs(semantic_recon_dir, exist_ok=True)
    return {
        "semantic_recon_dir": semantic_recon_dir,
        "semantic_recons_path": f"{OUTPUT_DIR}/{subject}_brainit_semantic_uint8_{FINAL_RECON_TAG}.pt",
        "semantic_ids_path": f"{OUTPUT_DIR}/{subject}_brainit_semantic_img_ids_{FINAL_RECON_TAG}.json",
        "enhanced_recons_path": f"{OUTPUT_DIR}/{subject}_brainit_enhanced_uint8_{FINAL_RECON_TAG}.pt",
        "enhanced_ids_path": f"{OUTPUT_DIR}/{subject}_brainit_enhanced_img_ids_{FINAL_RECON_TAG}.json",
    }


def semantic_part_path(paths, i):
    return f"{paths['semantic_recon_dir']}/img_{i:04d}.pt"


def semantic_ensemble_part_path(paths, i):
    return f"{paths['semantic_recon_dir']}/img_{i:04d}_ensemble.pt"


def ensure_combined_model_on_device():
    global combined_model
    if "_git_img2img_pipe" in globals() and _git_img2img_pipe is not None:
        try:
            _git_img2img_pipe.to("cpu")
        except Exception as exc:
            print("could not move GIT img2img pipe to CPU before semantic diffusion:", repr(exc))
    combined_model = combined_model.to(DEVICE).eval().requires_grad_(False)
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


def load_diffusion_inputs_for_subject(subject):
    if subject not in dip_manifest:
        raise KeyError(f"missing low-level DIP manifest for {subject}; run the DIP inversion cell first")
    lowlevel_paths = dip_manifest[subject]
    lowlevel_images = torch_load_cpu(lowlevel_paths["recons_path"]).float().clamp(0, 1)
    with open(lowlevel_paths["img_ids_path"]) as f:
        lowlevel_img_ids = [int(i) for i in json.load(f)]
    lowlevel_img_ids = lowlevel_img_ids[:len(lowlevel_images)]
    if any(img_id not in FINAL_HELDOUT_IMG_IDS for img_id in lowlevel_img_ids):
        raise ValueError(f"{subject} low-level diffusion inputs include non-heldout image IDs")

    pred = load_subject_prediction_for_lowlevel(subject)
    shared_img_ids = [int(i) for i in pred["shared_img_ids"]]
    if any(img_id not in FINAL_HELDOUT_IMG_IDS for img_id in shared_img_ids):
        raise ValueError(f"{subject} prediction cache includes non-heldout image IDs")
    clip_rows = _rows_for_ids(shared_img_ids, lowlevel_img_ids, f"{subject} predicted CLIP tokens")
    openclip_tokens = pred["pred_clip_tokens"][clip_rows].float().contiguous()
    expected = (CLIP_NUM_QUERIES, CLIP_DIM)
    if tuple(openclip_tokens.shape[1:]) != expected:
        raise ValueError(f"expected CLIP tokens [B, {expected[0]}, {expected[1]}], got {tuple(openclip_tokens.shape)}")
    if len(openclip_tokens) != len(lowlevel_images):
        raise ValueError(f"{subject} low-level/CLIP row mismatch: lowlevel={len(lowlevel_images)} clip={len(openclip_tokens)}")
    return lowlevel_images.contiguous(), openclip_tokens, lowlevel_img_ids


def select_best_semantic_candidates(indices, candidates_uint8, lowlevel_images):
    if DIFFUSION_ENSEMBLE_SIZE == 1:
        return candidates_uint8, None, None
    n = len(indices)
    candidates = candidates_uint8.reshape(n, DIFFUSION_ENSEMBLE_SIZE, *candidates_uint8.shape[1:])
    ref = lowlevel_images[indices].float().clamp(0, 1)
    ref = F.interpolate(ref, size=candidates.shape[-2:], mode="bicubic", align_corners=False)
    scores = (candidates.float().div(255.0) - ref[:, None]).square().mean(dim=(2, 3, 4))
    best = scores.argmin(dim=1)
    selected = candidates[torch.arange(n), best].contiguous()
    return selected, best.cpu(), scores.cpu()


@torch.inference_mode()
def generate_semantic_batch(indices, openclip_tokens, lowlevel_images):
    ensure_combined_model_on_device()
    idx = torch.as_tensor(indices, dtype=torch.long)
    clip_tokens = openclip_tokens[idx].to(DEVICE, non_blocking=True).float()
    if not USE_CLIP_TOKENS_FOR_DIFFUSION:
        clip_tokens = torch.zeros_like(clip_tokens)
    img_init = None
    if USE_LOWLEVEL_IMAGE_INIT_FOR_DIFFUSION:
        img_init = lowlevel_images[idx].permute(0, 2, 3, 1).float().clamp(0, 1).cpu().numpy()
    batch_size = int(clip_tokens.shape[0])
    dummy = torch.empty(batch_size, device=DEVICE, dtype=torch.long)
    samples = combined_model.generate(
        clip_tokens,
        dummy,
        dummy,
        img_init=img_init,
        num_samples=batch_size,
        start_step=DIFFUSION_START_STEP,
        num_steps=DIFFUSION_NUM_STEPS,
    )
    return samples.float().clamp(0, 1).cpu().mul(255).round().byte()


def generate_semantic_batch_safe(indices, openclip_tokens, lowlevel_images):
    try:
        return generate_semantic_batch(indices, openclip_tokens, lowlevel_images)
    except torch.cuda.OutOfMemoryError:
        if not FINAL_DIFFUSION_AUTO_SPLIT_ON_OOM or len(indices) <= 1:
            raise
        torch.cuda.empty_cache()
        mid = len(indices) // 2
        return torch.cat([
            generate_semantic_batch_safe(indices[:mid], openclip_tokens, lowlevel_images),
            generate_semantic_batch_safe(indices[mid:], openclip_tokens, lowlevel_images),
        ], dim=0)


def load_semantic_cache_if_valid(paths, diffusion_img_ids, final_diffusion_n):
    if not os.path.exists(paths["semantic_recons_path"]) or RERUN_FINAL_DIFFUSION:
        return None
    cached = torch_load_cpu(paths["semantic_recons_path"])
    cached_ids = None
    if os.path.exists(paths["semantic_ids_path"]):
        with open(paths["semantic_ids_path"]) as f:
            cached_ids = [int(i) for i in json.load(f)]
    if cached_ids is None or not _same_ids(cached_ids, diffusion_img_ids) or len(cached) != final_diffusion_n:
        print("rejecting stale semantic diffusion cache:", {
            "cache_rows": len(cached),
            "cache_ids": None if cached_ids is None else len(cached_ids),
            "needed": final_diffusion_n,
            "path": paths["semantic_recons_path"],
        })
        return None
    return cached


def diffuse_subject(subject):
    lowlevel_images, openclip_tokens, diffusion_img_ids = load_diffusion_inputs_for_subject(subject)
    paths = final_diffusion_paths_for_subject(subject)
    final_diffusion_n = len(diffusion_img_ids)
    print("diffusion conditioning aligned:", {
        "subject": subject,
        "n": final_diffusion_n,
        "first_ids": diffusion_img_ids[:8],
        "clip_shape": tuple(openclip_tokens.shape),
        "lowlevel_shape": tuple(lowlevel_images.shape),
        "clip_mean": float(openclip_tokens.float().mean()),
        "clip_std": float(openclip_tokens.float().std()),
    })

    semantic_uint8 = load_semantic_cache_if_valid(paths, diffusion_img_ids, final_diffusion_n)
    semantic_parts_need_full_rerun = bool(RERUN_FINAL_DIFFUSION)
    if semantic_uint8 is None:
        if os.path.exists(paths["semantic_ids_path"]):
            with open(paths["semantic_ids_path"]) as f:
                part_cache_ids = [int(i) for i in json.load(f)]
            if not _same_ids(part_cache_ids, diffusion_img_ids):
                print("rejecting stale semantic shard cache:", {
                    "part_cache_ids": len(part_cache_ids),
                    "needed": final_diffusion_n,
                    "path": paths["semantic_recon_dir"],
                })
                semantic_parts_need_full_rerun = True
        elif any(os.path.exists(semantic_part_path(paths, i)) for i in range(min(final_diffusion_n, 64))):
            print("semantic shards have no image-id sidecar; regenerating shards for current IDs.")
            semantic_parts_need_full_rerun = True
        pending = [i for i in range(final_diffusion_n) if semantic_parts_need_full_rerun or not os.path.exists(semantic_part_path(paths, i))]
        batch_size = max(1, int(FINAL_DIFFUSION_BATCH_SIZE))
        image_batch_size = max(1, batch_size // DIFFUSION_ENSEMBLE_SIZE)
        for start in tqdm(range(0, len(pending), image_batch_size), desc=f"BrainIT semantic diffusion {subject}"):
            batch_indices = pending[start:start + image_batch_size]
            candidate_indices = [i for i in batch_indices for _ in range(DIFFUSION_ENSEMBLE_SIZE)]
            candidates = generate_semantic_batch_safe(candidate_indices, openclip_tokens, lowlevel_images)
            samples, best_ids, scores = select_best_semantic_candidates(batch_indices, candidates, lowlevel_images)
            for i, sample in zip(batch_indices, samples):
                torch.save(sample, semantic_part_path(paths, i))
            if SAVE_DIFFUSION_ENSEMBLE_CANDIDATES and DIFFUSION_ENSEMBLE_SIZE > 1:
                candidates_by_image = candidates.reshape(len(batch_indices), DIFFUSION_ENSEMBLE_SIZE, *candidates.shape[1:])
                for row, i in enumerate(batch_indices):
                    torch.save({
                        "img_id": int(diffusion_img_ids[i]),
                        "best": int(best_ids[row]),
                        "scores": scores[row],
                        "candidates": candidates_by_image[row],
                    }, semantic_ensemble_part_path(paths, i))
            if DEVICE.type == "cuda" and FINAL_DIFFUSION_EMPTY_CACHE_EVERY and start % (image_batch_size * FINAL_DIFFUSION_EMPTY_CACHE_EVERY) == 0:
                torch.cuda.empty_cache()
        semantic_uint8 = torch.stack([torch_load_cpu(semantic_part_path(paths, i)) for i in range(final_diffusion_n)]).contiguous()
        torch.save(semantic_uint8, paths["semantic_recons_path"])
        with open(paths["semantic_ids_path"], "w") as f:
            json.dump(diffusion_img_ids, f)
    print("semantic reconstructions:", subject, tuple(semantic_uint8.shape), semantic_uint8.dtype, paths["semantic_recons_path"])

    if RUN_BRAINIT_SDXL_ENHANCEMENT:
        if os.path.exists(paths["enhanced_recons_path"]) and not RERUN_FINAL_DIFFUSION:
            final_uint8 = torch_load_cpu(paths["enhanced_recons_path"])
        else:
            semantic_hwc = semantic_uint8.float().div(255.0).permute(0, 2, 3, 1).numpy()
            enhanced_hwc = enhance_recons(
                semantic_hwc,
                unclip_config_path=MINDEYE2_UNCLIP_CONFIG,
                sdxl_config_path=BRAINIT_SDXL_CONFIG,
                sdxl_ckpt_path=BRAINIT_SDXL_CKPT_PATH,
                num_samples=1,
            )
            enhanced = torch.from_numpy(enhanced_hwc).permute(0, 3, 1, 2).float().clamp(0, 1)
            final_uint8 = F.interpolate(enhanced, size=(224, 224), mode="bicubic", align_corners=False).clamp(0, 1).mul(255).round().byte().contiguous()
            torch.save(final_uint8, paths["enhanced_recons_path"])
            with open(paths["enhanced_ids_path"], "w") as f:
                json.dump(diffusion_img_ids, f)
        final_recons_path = paths["enhanced_recons_path"]
        final_ids_path = paths["enhanced_ids_path"]
    else:
        final_uint8 = semantic_uint8
        final_recons_path = paths["semantic_recons_path"]
        final_ids_path = paths["semantic_ids_path"]

    print("final reconstructions:", subject, tuple(final_uint8.shape), final_uint8.dtype, final_recons_path)
    return {
        "subject": subject,
        "n": final_diffusion_n,
        "final_recon_tag": FINAL_RECON_TAG,
        "final_recons_path": final_recons_path,
        "final_ids_path": final_ids_path,
        "lowlevel_recons_path": dip_manifest[subject]["recons_path"],
        "lowlevel_ids_path": dip_manifest[subject]["img_ids_path"],
        "pred_clip_path": pred_clip_path(subject),
        **paths,
    }


final_diffusion_manifest = {subject: diffuse_subject(subject) for subject in FINAL_EVAL_SUBJECTS}
FINAL_DIFFUSION_MANIFEST_PATH = f"{OUTPUT_DIR}/final_diffusion_manifest_{FINAL_RECON_TAG}.json"
with open(FINAL_DIFFUSION_MANIFEST_PATH, "w") as f:
    json.dump(final_diffusion_manifest, f, indent=2)
print("saved final diffusion manifest:", FINAL_DIFFUSION_MANIFEST_PATH)


## 11. Final Evaluation, Image Export, and Preview


In [ ]:
# Final evaluation table and preview grids copied from final-eval and adapted per subject.
from torchvision import transforms
from torchvision.utils import save_image
from torchvision.models import (
    alexnet,
    AlexNet_Weights,
    efficientnet_b1,
    EfficientNet_B1_Weights,
    inception_v3,
    Inception_V3_Weights,
)
from torchvision.models.feature_extraction import create_feature_extractor
import open_clip

PREVIEW_SAMPLES_PER_COLLAGE = 48
PREVIEW_GRID_N = PREVIEW_SAMPLES_PER_COLLAGE
PREVIEW_GRID_COLS = 6
PREVIEW_GRID_DIR = f"{OUTPUT_DIR}/brainit_collage_previews_{FINAL_RECON_TAG}"
os.makedirs(PREVIEW_GRID_DIR, exist_ok=True)



IMAGE_EXPORT_ROOT = f"{OUTPUT_DIR}/drive_image_exports_{FINAL_RECON_TAG}"
os.makedirs(IMAGE_EXPORT_ROOT, exist_ok=True)


def save_subject_image_exports(subject, final_img_ids, true_images, low_images, final_images):
    subject_dir = f"{IMAGE_EXPORT_ROOT}/{subject}"
    dirs = {
        "actual": f"{subject_dir}/actual",
        "vgg_dip": f"{subject_dir}/vgg_dip_preview",
        "final_diffusion": f"{subject_dir}/final_diffusion",
    }
    for d in dirs.values():
        os.makedirs(d, exist_ok=True)
    for row, img_id in enumerate(final_img_ids):
        stem = f"{row:04d}_img{int(img_id):06d}"
        save_image(true_images[row].clamp(0, 1), f"{dirs['actual']}/{stem}.png")
        save_image(low_images[row].clamp(0, 1), f"{dirs['vgg_dip']}/{stem}.png")
        save_image(final_images[row].clamp(0, 1), f"{dirs['final_diffusion']}/{stem}.png")
    return subject_dir


def norm_batch(x, size, mean, std):
    x = F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False)
    mean = torch.tensor(mean, device=x.device).view(1, 3, 1, 1)
    std = torch.tensor(std, device=x.device).view(1, 3, 1, 1)
    return (x - mean) / std


@torch.no_grad()
def extract_features(images, fn, batch_size=FINAL_EVAL_FEATURE_BATCH_SIZE):
    outs = []
    for start in tqdm(range(0, len(images), batch_size), desc="feature batches", leave=False):
        outs.append(fn(images[start:start + batch_size].to(DEVICE)).float().flatten(1).cpu())
    return torch.cat(outs, dim=0)


def two_way_score(recon_feats, true_feats):
    r = np.corrcoef(true_feats.numpy(), recon_feats.numpy())[:len(true_feats), len(true_feats):]
    congruent = np.diag(r)
    return float(np.mean(np.sum(r < congruent, axis=0)) / (len(true_feats) - 1))


def correlation_distance(recon_feats, true_feats):
    a, b = recon_feats.numpy(), true_feats.numpy()
    return float(np.mean([1.0 - np.corrcoef(a[i], b[i])[0, 1] for i in range(len(a))]))


def retrieval_score(pred_feats, true_feats, n_loops=30, n_samples=300, seed=SEED):
    pred = F.normalize(pred_feats.float(), dim=-1)
    true = F.normalize(true_feats.float(), dim=-1)
    n = len(pred)
    fwd, bwd = [], []
    rng = np.random.RandomState(seed)
    for _ in range(n_loops):
        idx = torch.as_tensor(rng.choice(n, size=min(n_samples, n), replace=False), dtype=torch.long)
        labels = torch.arange(len(idx))
        fwd.append((pred[idx] @ true[idx].T).argmax(1).eq(labels).float().mean().item())
        bwd.append((true[idx] @ pred[idx].T).argmax(1).eq(labels).float().mean().item())
    return float(np.mean(fwd)), float(np.mean(bwd))


def pixel_metrics(recons, images):
    resized_recons = F.interpolate(recons, size=(425, 425), mode="bilinear", align_corners=False)
    resized_images = F.interpolate(images, size=(425, 425), mode="bilinear", align_corners=False)
    pixcorr = np.mean([
        np.corrcoef(a.flatten(), b.flatten())[0, 1]
        for a, b in zip(resized_recons.numpy(), resized_images.numpy())
    ])
    rec_gray = rgb2gray(resized_recons.permute(0, 2, 3, 1).numpy())
    img_gray = rgb2gray(resized_images.permute(0, 2, 3, 1).numpy())
    ssim = np.mean([
        ssim_fn(r, i, data_range=1.0, gaussian_weights=True, sigma=1.5, use_sample_covariance=False)
        for r, i in zip(rec_gray, img_gray)
    ])
    return float(pixcorr), float(ssim)


def load_true_clip_targets(subject, img_ids):
    cache_path = f"{CLIP_TARGETS_DIR}/{subject}_clip_targets.pt"
    if not os.path.isfile(cache_path):
        return None
    cache = torch_load_cpu(cache_path)
    if isinstance(cache, dict):
        vals = [cache.get(int(i)) for i in img_ids]
        if any(v is None for v in vals):
            return None
        return torch.stack([v.float() for v in vals])
    return None


def load_pred_clip_tokens_for_ids(subject, img_ids):
    pred = load_subject_prediction_for_lowlevel(subject)
    rows = _rows_for_ids(pred["shared_img_ids"], img_ids, f"{subject} predicted CLIP tokens")
    return pred["pred_clip_tokens"][rows].float().contiguous()


def load_lowlevel_images_for_ids(paths, img_ids):
    low = torch_load_cpu(paths["lowlevel_recons_path"]).float().clamp(0, 1)
    with open(paths["lowlevel_ids_path"]) as f:
        low_ids = [int(i) for i in json.load(f)]
    rows = _rows_for_ids(low_ids, img_ids, "low-level DIP images")
    return low[rows].contiguous()


def save_brainit_preview_grid(subject, final_img_ids, true_images, low_images, final_images, indices, path, show=False):
    indices = [int(i) for i in indices]
    if len(indices) > PREVIEW_SAMPLES_PER_COLLAGE:
        raise ValueError(f"preview collage requested {len(indices)} samples; max is {PREVIEW_SAMPLES_PER_COLLAGE}")
    if not indices:
        return None

    true_img = true_images[indices]
    low_img = F.interpolate(low_images[indices], size=final_images.shape[-1], mode="bicubic", align_corners=False)
    final_img = final_images[indices]
    combined = torch.cat([true_img, low_img, final_img], dim=3).clamp(0, 1)

    n_show = len(indices)
    cols = min(PREVIEW_GRID_COLS, n_show)
    rows_n = int(np.ceil(n_show / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(7.2 * cols, 3.0 * rows_n), constrained_layout=True)
    axes = np.atleast_1d(axes).reshape(rows_n, cols)
    for ax in axes.ravel():
        ax.axis("off")
    for k, idx in enumerate(indices):
        ax = axes[k // cols, k % cols]
        ax.imshow(combined[k].permute(1, 2, 0))
        ax.set_title(f"{subject} id {final_img_ids[idx]}: true | low | final", fontsize=8)
    fig.savefig(path, dpi=180, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)
    return path


def save_all_brainit_preview_grids(subject, final_img_ids, true_images, low_images, final_images):
    paths = []
    n = len(final_images)
    for part, start in enumerate(range(0, n, PREVIEW_SAMPLES_PER_COLLAGE), start=1):
        stop = min(start + PREVIEW_SAMPLES_PER_COLLAGE, n)
        path = f"{PREVIEW_GRID_DIR}/{subject}_brainit_collage_preview_{FINAL_RECON_TAG}_part{part:03d}_{start:04d}-{stop - 1:04d}.png"
        paths.append(save_brainit_preview_grid(
            subject,
            final_img_ids,
            true_images,
            low_images,
            final_images,
            range(start, stop),
            path,
            show=False,
        ))
    return paths


alex = create_feature_extractor(
    alexnet(weights=AlexNet_Weights.IMAGENET1K_V1),
    return_nodes={"features.4": "a2", "features.11": "a5"},
).to(DEVICE).eval().requires_grad_(False)
inception = create_feature_extractor(
    inception_v3(weights=Inception_V3_Weights.DEFAULT), return_nodes={"avgpool": "pool"}
).to(DEVICE).eval().requires_grad_(False)
effnet = create_feature_extractor(
    efficientnet_b1(weights=EfficientNet_B1_Weights.DEFAULT), return_nodes={"avgpool": "pool"}
).to(DEVICE).eval().requires_grad_(False)
clip_eval, _, _ = open_clip.create_model_and_transforms("ViT-L-14", pretrained="openai", device=DEVICE)
clip_eval = clip_eval.eval().requires_grad_(False)
swav = (
    create_feature_extractor(
        torch.hub.load("facebookresearch/swav:main", "resnet50"), return_nodes={"avgpool": "pool"}
    ).to(DEVICE).eval().requires_grad_(False)
    if RUN_SWAV_EVAL
    else None
)

imagenet_mean, imagenet_std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
clip_mean, clip_std = [0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711]


def alex_feats(x, key):
    return extract_features(x, lambda b: alex(norm_batch(b, 256, imagenet_mean, imagenet_std))[key])


def inception_feats(x):
    return extract_features(x, lambda b: inception(norm_batch(b, 342, imagenet_mean, imagenet_std))["pool"])


def eff_feats(x):
    return extract_features(x, lambda b: effnet(norm_batch(b, 255, imagenet_mean, imagenet_std))["pool"])


def clip_feats(x):
    return extract_features(x, lambda b: clip_eval.encode_image(norm_batch(b, 224, clip_mean, clip_std)))


def swav_feats(x):
    return extract_features(x, lambda b: swav(norm_batch(b, 224, imagenet_mean, imagenet_std))["pool"])


def evaluate_final_subject(subject, paths):
    final_uint8 = torch_load_cpu(paths["final_recons_path"])
    with open(paths["final_ids_path"]) as f:
        final_img_ids = [int(i) for i in json.load(f)]
    final_img_ids = final_img_ids[:len(final_uint8)]
    if any(img_id not in FINAL_HELDOUT_IMG_IDS for img_id in final_img_ids):
        raise ValueError(f"{subject} final eval IDs include non-heldout image IDs")

    true_images = images_for_ids(final_img_ids, size=256).float()
    final_images = final_uint8.float().div(255.0)
    if final_images.shape[-1] != 256:
        final_images = F.interpolate(final_images, size=(256, 256), mode="bicubic", align_corners=False).clamp(0, 1)
    low_images = F.interpolate(load_lowlevel_images_for_ids(paths, final_img_ids), size=(256, 256), mode="bicubic", align_corners=False).clamp(0, 1)
    export_dir = save_subject_image_exports(subject, final_img_ids, true_images, low_images, final_images)
    print("saved per-image PNG exports:", export_dir)

    true_feature_cache = {
        "alex2": alex_feats(true_images, "a2"),
        "alex5": alex_feats(true_images, "a5"),
        "inception": inception_feats(true_images),
        "clip": clip_feats(true_images),
        "eff": eff_feats(true_images),
    }
    if RUN_SWAV_EVAL:
        true_feature_cache["swav"] = swav_feats(true_images)

    rows = []
    for method, recons in [("Low-Level", low_images), ("Final", final_images)]:
        pixcorr, ssim = pixel_metrics(recons.cpu(), true_images.cpu())
        recon_clip = clip_feats(recons)
        row = {
            "Subject": subject,
            "Method": method,
            "N": len(recons),
            "PixCorr": pixcorr,
            "SSIM": ssim,
            "AlexNet(2)": two_way_score(alex_feats(recons, "a2"), true_feature_cache["alex2"]),
            "AlexNet(5)": two_way_score(alex_feats(recons, "a5"), true_feature_cache["alex5"]),
            "InceptionV3": two_way_score(inception_feats(recons), true_feature_cache["inception"]),
            "CLIP": two_way_score(recon_clip, true_feature_cache["clip"]),
            "EffNet-B": correlation_distance(eff_feats(recons), true_feature_cache["eff"]),
        }
        if RUN_SWAV_EVAL:
            row["SwAV"] = correlation_distance(swav_feats(recons), true_feature_cache["swav"])
        row["ImageRetrieval"], row["BrainRetrieval"] = retrieval_score(
            recon_clip, true_feature_cache["clip"], n_samples=min(300, len(recons)), seed=SEED + SUBJECT_TO_IDX[subject]
        )
        rows.append(row)

    true_semantic = load_true_clip_targets(subject, final_img_ids)
    pred_semantic = load_pred_clip_tokens_for_ids(subject, final_img_ids)
    if true_semantic is not None and tuple(true_semantic.shape[-2:]) == tuple(pred_semantic.shape[-2:]):
        semantic_pred = pred_semantic.float().mean(1)
        semantic_true = true_semantic.float().mean(1)
        rows[-1]["SemanticImageRetrieval"], rows[-1]["SemanticBrainRetrieval"] = retrieval_score(
            semantic_pred, semantic_true, n_samples=min(300, len(semantic_pred)), seed=SEED + SUBJECT_TO_IDX[subject]
        )
    else:
        print("semantic retrieval skipped: missing/mismatched", subject)

    metrics_df = pd.DataFrame(rows).set_index("Method")
    final_eval_path = f"{OUTPUT_DIR}/{subject}_mindeye2_table_metrics_{FINAL_RECON_TAG}.csv"
    metrics_df.to_csv(final_eval_path)
    display(metrics_df)
    print("saved:", final_eval_path)

    n_preview = min(PREVIEW_GRID_N, len(final_images))
    rng = np.random.RandomState(SEED + SUBJECT_TO_IDX[subject])
    preview_idx = rng.choice(len(final_images), size=n_preview, replace=False)
    preview_grid_path = f"{PREVIEW_GRID_DIR}/{subject}_brainit_sample48_preview_{FINAL_RECON_TAG}.png"
    save_brainit_preview_grid(
        subject,
        final_img_ids,
        true_images,
        low_images,
        final_images,
        preview_idx,
        preview_grid_path,
        show=True,
    )
    print("saved sample preview:", preview_grid_path)

    all_preview_paths = save_all_brainit_preview_grids(subject, final_img_ids, true_images, low_images, final_images)
    print(f"saved {len(all_preview_paths)} all-image preview collage(s) for {subject}:", PREVIEW_GRID_DIR)
    return metrics_df.reset_index()


if "final_diffusion_manifest" not in globals():
    with open(FINAL_DIFFUSION_MANIFEST_PATH) as f:
        final_diffusion_manifest = json.load(f)

final_eval_tables = [evaluate_final_subject(subject, final_diffusion_manifest[subject]) for subject in FINAL_EVAL_SUBJECTS]
final_eval_summary = pd.concat(final_eval_tables, ignore_index=True)
FINAL_TABLE_SUMMARY_PATH = f"{OUTPUT_DIR}/mindeye2_table_metrics_summary_{FINAL_RECON_TAG}.csv"
final_eval_summary.to_csv(FINAL_TABLE_SUMMARY_PATH, index=False)
display(final_eval_summary)
print("saved summary:", FINAL_TABLE_SUMMARY_PATH)
